In [78]:
#Formats inputs of motifs into convenient format
#Input of Stem is N1, e1; N2, e2; ..., where ei = 1 iff Ni is basepaired in the stem; read 5' to 3'
def insertStem():
    leftstem = []
    rightstem = []
    print("Left stem interactions (5' to 3')?")
    lInput = input()
    if len(lInput) > 0:
        lInput = lInput.split(';')
        for el in lInput:
            el = el.split(',')
            el = [el[0], int(el[1])]
            leftstem.append(el)
    print("Right stem interactions (5' to 3')?")
    rInput = input()
    if len(rInput) > 0:
        rInput = rInput.split(';')
        for el in rInput:
            el = el.split(',')
            el = [el[0], int(el[1])]
            rightstem.append(el)
    leftstemHMMs = []
    rightstemHMMs = []
    if len(leftstem) == 0:
        leftstemHMMs.append(['', 0])
    else:
        block = ''
        for i in range(0, len(leftstem)):
            if leftstem[i][1] == 0:
                if i < len(leftstem)-1:
                    block+=leftstem[i][0]
                else:
                    block+=leftstem[i][0]
                    leftstemHMMs.append([block, 0])
            else:
                if block != "":
                    leftstemHMMs.append([block, 0])
                leftstemHMMs.append([leftstem[i][0], 1])
                block = ''
    if len(rightstem) == 0:
        rightstemHMMs.append(['', 0])
    else:
        block = ''
        for i in range(0, len(rightstem)):
            if rightstem[i][1] == 0:
                if i < len(rightstem)-1:
                    block+=rightstem[i][0]
                else:
                    block+=rightstem[i][0]
                    rightstemHMMs.append([block, 0])
            else:
                if block != "":
                    rightstemHMMs.append([block, 0])
                rightstemHMMs.append([rightstem[i][0], 1])
                block = ''
    print(leftstemHMMs)
    print(rightstemHMMs)
    if leftstemHMMs[0][1] == 1 and rightstemHMMs[len(rightstemHMMs)-1][1] == 0:
        leftstemHMMs.insert(0, ['', 0])
    if leftstemHMMs[0][1] == 0 and rightstemHMMs[len(rightstemHMMs)-1][1] == 1:
        rightstemHMMs.append(['', 0])
    if leftstemHMMs[len(leftstemHMMs)-1][1] == 1 and rightstemHMMs[0][1] == 0:
        leftstemHMMs.append(['', 0])
    if leftstemHMMs[len(leftstemHMMs)-1][1] == 0 and rightstemHMMs[0][1] == 1:
        rightstemHMMs.insert(0, ['', 0])
    return leftstemHMMs, rightstemHMMs



HLPacket = []
print("How many Hairpin Loop Motifs?")
num_HLMotifs = int(input()) 
if num_HLMotifs > 0:
    for i in range(0, num_HLMotifs):
        print("Loop consensus sequence (5' to 3')?")
        loopConsensus = str(input())
        if loopConsensus == "":
            loopConsensus = "Bulge"
        leftHLstem, rightHLstem = insertStem()
        HLPacket.append([loopConsensus, [leftHLstem, rightHLstem]])
        
        
ILPacket = []
print("How many Internal Loop Motifs?")
num_ILMotifs = int(input())
if num_ILMotifs > 0:
    for i in range(0, num_ILMotifs):
        print("Inner stem: ")
        leftlowerILstem, rightlowerILstem = insertStem()
        print("Left bulge consensus sequence (5' to 3')?")
        leftbulge = str(input())
        if leftbulge == "":
            leftbulge = "Bulge"
        print("Right bulge consensus sequence (5' to 3')?")
        rightbulge = str(input())
        if rightbulge == "":
            rightbulge = "Bulge"
        print("Outer stem: ")
        leftupperILstem, rightupperILstem = insertStem()
        ILPacket.append([[leftlowerILstem, rightlowerILstem], [leftbulge, rightbulge], [leftupperILstem, rightupperILstem]])

    
print(HLPacket)
print(ILPacket)


How many Hairpin Loop Motifs?
1
Loop consensus sequence (5' to 3')?
GNRA
Left stem interactions (5' to 3')?

Right stem interactions (5' to 3')?

[['', 0]]
[['', 0]]
How many Internal Loop Motifs?
1
Inner stem: 
Left stem interactions (5' to 3')?
N,1;G,0;A,0
Right stem interactions (5' to 3')?
G,0;A,0;N,1
[['N', 1], ['GA', 0]]
[['GA', 0], ['N', 1]]
Left bulge consensus sequence (5' to 3')?

Right bulge consensus sequence (5' to 3')?
RNN
Outer stem: 
Left stem interactions (5' to 3')?

Right stem interactions (5' to 3')?

[['', 0]]
[['', 0]]
[['GNRA', [[['', 0]], [['', 0]]]]]
[[[[['N', 1], ['GA', 0]], [['GA', 0], ['N', 1]]], ['Bulge', 'RNN'], [[['', 0]], [['', 0]]]]]


In [79]:
#Reverse orientation for IL motifs, since 5' and 3' ordering can still be satisfied up to a 180* rotation

def reverseMotif(motif):
    inPacket = [motif[2][1], motif[2][0]]
    outPacket = [motif[0][1], motif[0][0]]
    bulges = [motif[1][1], motif[1][0]]
    revMotif = [inPacket, bulges, outPacket]
    return revMotif

for motif in range(0, len(ILPacket)):
    ILPacket.append(reverseMotif(ILPacket[motif]))
    
print(ILPacket)
num_ILMotifs = len(ILPacket)


[[[[['N', 1], ['GA', 0]], [['GA', 0], ['N', 1]]], ['Bulge', 'RNN'], [[['', 0]], [['', 0]]]], [[[['', 0]], [['', 0]]], ['RNN', 'Bulge'], [[['GA', 0], ['N', 1]], [['N', 1], ['GA', 0]]]]]


In [80]:
#Per HL Motif, counting the number of stem elements +1 (for convenience)

num_HLStem = []

if num_HLMotifs > 0:
    for t in range(0, num_HLMotifs):
        if HLPacket[t][1][0][0][0] == '' and HLPacket[t][1][1][0][0] == '':
            num_HLStem.append(1)
        else:
            num_HLStem.append(len(HLPacket[t][1][0]) + 1)
            
print(num_HLStem)



[1]


In [81]:
#Per IL In Stem (before the bulges), counts 1 + length of stem

num_ILInStem = []

if num_ILMotifs > 0:
    for t in range(0, num_ILMotifs):
        if ILPacket[t][0][0][0][0] == '' and ILPacket[t][0][1][0][0] == '':
            num_ILInStem.append(1)
        else:
            num_ILInStem.append(len(ILPacket[t][0][0]) + 1)
            
print(num_ILInStem)


[3, 1]


In [82]:
#Per IL Out Stem, counts length of Stem

num_ILOutStem = []

if num_ILMotifs > 0:
    for t in range(0, num_ILMotifs):
        if ILPacket[t][2][0][0][0] == '' and ILPacket[t][2][1][0][0] == '':
            num_ILOutStem.append(0)
        else:
            num_ILOutStem.append(len(ILPacket[t][2][0]))
            
print(num_ILOutStem)

[0, 2]


In [83]:
bpcut_HLStem = []

if num_HLMotifs > 0:
    for t in range(0, num_HLMotifs):
        cutoff = 1
        for g in range(len(HLPacket[t][1][0])-1, -1, -1):
            if HLPacket[t][1][0][g][1] == 0:
                cutoff+=1
            else:
                break
        bpcut_HLStem.append(cutoff)
print(bpcut_HLStem)

[2]


In [84]:
import math

pa = [math.log(0.991), math.log(0.003), math.log(0.003), math.log(0.003)]
pg = [math.log(0.001), math.log(0.997), math.log(0.001), math.log(0.001)]
pc = [math.log(0.001), math.log(0.001), math.log(0.997), math.log(0.001)]
pu = [math.log(0.001), math.log(0.001), math.log(0.001), math.log(0.997)]

pw = [math.log(0.499), math.log(0.001), math.log(0.001), math.log(0.499)]
ps = [math.log(0.001), math.log(0.499), math.log(0.499), math.log(0.001)]
pr = [math.log(0.499), math.log(0.499), math.log(0.001), math.log(0.001)]
py = [math.log(0.001), math.log(0.001), math.log(0.499), math.log(0.499)]
pm = [math.log(0.499), math.log(0.001), math.log(0.499), math.log(0.001)]
pk = [math.log(0.001), math.log(0.499), math.log(0.001), math.log(0.499)]

pb = [math.log(0.001), math.log(0.333), math.log(0.333), math.log(0.333)]
pd = [math.log(0.333), math.log(0.001), math.log(0.333), math.log(0.333)]
ph = [math.log(0.333), math.log(0.333), math.log(0.001), math.log(0.333)]
pv = [math.log(0.333), math.log(0.333), math.log(0.333), math.log(0.001)]

pn = [-1.016725, -1.500931, -1.729724, -1.435626]



letter_emis = {"A" : pa, "G" : pg, "C" : pc, "U" : pu, "W" : pw, "S" : ps, "M" : pm, "K" : pk, "R" : pr, "Y" : py, "B" : pb, "D" : pd, "H" : ph, "V" : pv, "N" : pn}
n_emissionsdict = {"A" : pn[0], "G" : pn[1], "C" : pn[2], "U" : pn[3]}


letter_emissionsdict = {}
for el in letter_emis:
    letter_emissionsdict[el] = {"A" : letter_emis[el][0], "G" : letter_emis[el][1], "C" : letter_emis[el][2], "U" : letter_emis[el][3]}

pIn = [math.log(0.005), math.log(0.995)]

pn = [-1.016725, -1.500931, -1.729724, -1.435626]
n_emissionsdict = {"A" : pn[0], "G" : pn[1], "C" : pn[2], "U" : pn[3]}



HLlendist = [-math.inf, -15.0, -8.903444, -5.903444, -2.903444, -1.209496, -2.142073, -1.983893, -1.861143, -2.620094, -2.911159, -3.311795, -3.764607, -4.256670, -4.860949, -4.916907, -4.938726, -5.123993, -5.256261, -5.676994, -6.067192, -5.617275, -5.939358, -6.450184, -6.162502, -6.267862, -6.715887, -7.212324, -6.632505, -7.654157, -7.771940, -7.366475]

phlsing = [-1.105321, -1.530578, -1.696738, -1.312284]

hlsing_emissionsdict = {"A" : phlsing[0], "G" : phlsing[1], "C" : phlsing[2], "U" : phlsing[3]}

pHLExt = [math.log(0.00006), math.log(0.99994)]
pILExt = [math.log(0.00006), math.log(0.99994)]



ILlendist = [math.log(0.995), math.log(0.004), math.log(0.0005), math.log(0.0003), math.log(0.0002)]
pbulgesing = [-0.827097, -1.658819, -1.954306, -1.466825]
pintloopsing = [-0.944122, -1.497648, -1.830329, -1.482969]

qsing_emissionsdict = {"A" : pbulgesing[0], "G" : pbulgesing[1], "C" : pbulgesing[2], "U" : pbulgesing[3]}
bulgesing_emissionsdict = {"A" : pbulgesing[0], "G" : pbulgesing[1], "C" : pbulgesing[2], "U" : pbulgesing[3]}

Pbulgedist = [-0.563112, -1.526062, -2.392811, -2.928284, -3.225183, -4.458385, -4.449946, -5.024377, -4.416886, -6.521020, -7.149628, -7.031845, -9.229070, -8.535923, -8.535923, -9.229070, -7.842776, -9.229070, -8.130458, -8.535923, -9.229070, -7.619632, -9.229070, -8.130458, -9.229070, -8.130458, -7.283160, -9.229070, -9.229070, -9.229070]


In [85]:
pS = [-0.223274, -2.139489, -2.496295]
pL = [-0.136271, -2.060474]
pF = [-0.353167, -1.212207]
pP = [-2.4220982943875504, -2.4220982943875504, -1.1335762943875507, -1.1002202943875505, -2.1383292943875505, math.log(0.05)]
pM = [-0.876334, -0.538379]
pR = [-0.238314, -1.550958]
pM1 = [-0.200071, -1.707450]
pbpL1 = [-6.299321, -6.431248, -5.777322, -1.953912, -6.405931, -6.937810, -1.355754, -6.896137, -6.143566, -1.148826, -6.553851, -2.372133, -2.030174, -6.357141, -3.197610, -5.959644]
pbpL2 = [-7.256579, -7.767405, -7.256579, -1.957263, -7.256579, -6.563432, -1.302817, -7.767405, -7.767405, -1.362728, -7.256579, -2.612189, -1.678360, -8.866017, -2.765698, -6.032804]


In [86]:
    
pbpL1 = [-6.299321, -6.431248, -5.777322, -1.953912, -6.405931, -6.937810, -1.355754, -6.896137, -6.143566, -1.148826, -6.553851, -2.372133, -2.030174, -6.357141, -3.197610, -5.959644]
pbpL2 = [-7.256579, -7.767405, -7.256579, -1.957263, -7.256579, -6.563432, -1.302817, -7.767405, -7.767405, -1.362728, -7.256579, -2.612189, -1.678360, -8.866017, -2.765698, -6.032804]

HLlendist = [-math.inf, -15.0, -8.903444, -5.903444, -2.903444, -1.209496, -2.142073, -1.983893, -1.861143, -2.620094, -2.911159, -3.311795, -3.764607, -4.256670, -4.860949, -4.916907, -4.938726, -5.123993, -5.256261, -5.676994, -6.067192, -5.617275, -5.939358, -6.450184, -6.162502, -6.267862, -6.715887, -7.212324, -6.632505, -7.654157, -7.771940, -7.366475]


phlsing = [-1.105321, -1.530578, -1.696738, -1.312284]
pbulgesing = [-0.827097, -1.658819, -1.954306, -1.466825]
pintloopsing = [-0.944122, -1.497648, -1.830329, -1.482969]

qsing_emissionsdict = {"A" : pbulgesing[0], "G" : pbulgesing[1], "C" : pbulgesing[2], "U" : pbulgesing[3]}
Qslendist = [math.log(0.995), math.log(0.004), math.log(0.0005), math.log(0.0003), math.log(0.0002)]

n_emissionsdict = {"A" : pn[0], "G" : pn[1], "C" : pn[2], "U" : pn[3]}
hlsing_emissionsdict = {"A" : phlsing[0], "G" : phlsing[1], "C" : phlsing[2], "U" : phlsing[3]}
bulgesing_emissionsdict = {"A" : pbulgesing[0], "G" : pbulgesing[1], "C" : pbulgesing[2], "U" : pbulgesing[3]}
intloopsing_emissionsdict = {"A" : pintloopsing[0], "G" : pintloopsing[1], "C" : pintloopsing[2], "U" : pintloopsing[3]}

g_emissionsdict = {"A" : pg[0], "G" : pg[1], "C" : pg[2], "U" : pg[3]}
r_emissionsdict = {"A" : pr[0], "G" : pr[1], "C" : pr[2], "U" : pr[3]}
a_emissionsdict = {"A" : pa[0], "G" : pa[1], "C" : pa[2], "U" : pa[3]}

stackedbpF1 = [ 
    [ -3.572346, -4.083171, -2.473733, -1.885947, -3.572346, -5.181784, -1.685276, -5.181784, -5.181784, -2.348570, -5.181784, -3.235873, -1.468211, -5.181784, -2.473733, -3.572346],  
    [ -2.765620, -3.864232, -2.765620, -2.765620, -3.864232, -3.576550, -1.827350, -4.962845, -3.864232, -1.031019, -4.269697, -4.269697, -2.564949, -4.962845, -3.864232, -2.765620],   
    [ -3.670861, -3.670861, -3.419547, -2.481277, -5.616771, -5.616771, -1.427116, -4.518159, -5.616771, -2.120264, -5.616771, -1.197930, -2.481277, -5.616771, -3.670861, -3.419547],   
    [ -7.287034, -8.242546, -7.906074, -1.486079, -7.654759, -9.851984, -1.577372, -7.906074, -7.454088, -1.337996, -7.018770, -3.009300, -1.613183, -7.454088, -2.961375, -8.242546],   
    [ -5.153292, -3.543854, -2.956067, -2.588342, -3.543854, -5.153292, -2.108769, -3.543854, -5.153292, -2.320078, -4.054679, -3.207381, -1.303144, -4.460144, -2.108769, -2.445241],   
    [ -4.330733, -3.232121, -3.232121, -2.721295, -3.637586, -3.637586, -1.932838, -4.330733, -4.330733, -2.251292, -2.721295, -4.330733, -2.944439, -3.232121, -2.384823, -1.386294],   
    [ -7.411059, -8.047047, -7.411059, -1.734305, -6.580710, -8.452512, -1.295816, -6.201221, -7.108778, -1.532170, -7.411059, -3.384657, -1.432471, -8.164830, -2.930385, -7.025396],   
    [ -2.425483, -4.990433, -2.045994, -2.793208, -3.891820, -4.297285, -1.694596, -2.425483, -4.990433, -1.694596, -4.990433, -3.380995, -2.592537, -4.990433, -4.990433, -2.425483],   
    [ -3.357395, -4.204693, -2.905410, -2.258782, -2.905410, -5.303305, -1.936009, -5.303305, -4.610158, -1.453157, -4.204693, -3.106080, -1.692387, -3.357395, -3.106080, -4.204693],   
    [ -7.486778, -7.922096, -6.606419, -2.074345, -8.710554, -10.319992, -1.390424, -10.319992, -6.764643, -1.074960, -7.275469, -2.784161, -1.944362, -8.017406, -2.603085, -7.611941],   
    [ -5.003946, -2.806722, -5.003946, -2.059507, -3.905334, -3.905334, -2.438997, -3.905334, -2.113575, -2.059507, -3.394508, -3.905334, -1.507439, -3.394508, -2.806722, -3.394508],   
    [ -7.962067, -7.114769, -7.962067, -2.003643, -9.060680, -7.962067, -1.169723, -9.060680, -6.662784, -1.193574, -7.114769, -2.511029, -1.975615, -9.060680, -3.618262, -7.451242],   
    [ -6.494973, -7.364011, -6.984521, -1.635661, -6.318042, -7.731736, -1.476839, -7.983050, -7.095747, -1.347104, -6.561665, -3.329090, -1.517795, -7.220910, -2.944244, -7.531065],   
    [ -3.725693, -5.111988, -4.013375, -2.278774, -2.914763, -4.013375, -1.893112, -5.111988, -3.502550, -1.199965, -4.013375, -4.013375, -1.678001, -4.013375, -5.111988, -3.166078],   
    [ -6.535564, -7.634176, -7.634176, -1.886908, -6.334893, -8.732788, -1.293229, -7.634176, -6.535564, -1.641879, -7.634176, -2.013775, -1.826034, -5.899575, -2.595061, -6.334893],   
    [ -4.454347, -4.454347, -3.943522, -2.118972, -5.552960, -4.859812, -1.746297, -5.552960, -3.943522, -1.075623, -3.943522, -3.607049, -2.257123, -3.607049, -3.155064, -2.719746],   
  ]

stackedbpF2 = [ 
    [ -2.258782, -4.204693, -4.204693, -3.106080, -4.204693, -3.106080, -3.106080, -3.511545, -2.258782, -0.908856, -4.204693, -4.204693, -2.595255, -4.204693, -3.106080, -4.204693], 
    [ -3.583519, -3.583519, -3.583519, -3.583519, -3.583519, -3.583519, -2.484907, -3.583519, -3.583519, -0.750306, -3.583519, -2.484907, -3.583519, -3.583519, -3.583519, -3.583519], 
    [ -3.060271, -4.158883, -4.158883, -4.158883, -4.158883, -4.158883, -1.023389, -2.549445, -4.158883, -1.214444, -4.158883, -4.158883, -3.060271, -4.158883, -4.158883, -4.158883],   
    [ -7.277386, -6.321874, -6.488929, -1.873808, -7.500529, -8.886824, -1.364424, -6.178774, -6.940914, -1.058387, -7.277386, -3.357395, -2.140412, -7.277386, -2.532454, -5.842301],   
    [ -4.204693, -4.204693, -4.204693, -4.204693, -2.258782, -4.204693, -0.985817, -4.204693, -4.204693, -1.806797, -4.204693, -3.511545, -2.258782, -3.106080, -3.106080, -4.204693],   
    [ -3.295837, -3.295837, -3.295837, -3.295837, -3.295837, -3.295837, -3.295837, -3.295837, -2.602690, -2.197225, -1.349927, -3.295837, -3.295837, -3.295837, -2.197225, -3.295837],   
    [ -7.268068, -6.632079, -6.757242, -2.072030, -6.287239, -9.465293, -0.843559, -8.772145, -8.366680, -1.641647, -7.673533, -3.570890, -2.051925, -7.519382, -2.446891, -7.519382],   
    [ -3.135494, -3.135494, -3.135494, -2.036882, -3.135494, -3.135494, -3.135494, -2.442347, -3.135494, -2.036882, -3.135494, -3.135494, -2.036882, -3.135494, -3.135494, -3.135494],   
    [ -4.127134, -4.127134, -3.028522, -3.028522, -3.028522, -2.047693, -2.047693, -2.517696, -4.127134, -3.028522, -4.127134, -4.127134, -1.419084, -4.127134, -2.181224, -4.127134],   
    [ -8.578414, -9.677026, -9.677026, -2.270315, -6.121678, -8.578414, -1.264083, -7.112077, -6.843813, -1.079359, -7.597584, -3.396630, -2.001480, -6.904437, -2.312479, -7.279131],   
    [ -4.060443, -4.060443, -4.060443, -1.863218, -2.114533, -1.981001, -1.352393, -4.060443, -4.060443, -4.060443, -2.674149, -4.060443, -4.060443, -3.367296, -3.367296, -2.961831],   
    [ -7.638198, -6.028760, -7.638198, -1.602717, -7.638198, -7.638198, -1.369102, -6.539586, -7.638198, -1.247958, -6.945051, -3.320710, -1.754876, -6.539586, -3.294393, -6.028760],   
    [ -7.844110, -6.109509, -6.996812, -1.910981, -7.844110, -7.844110, -1.148311, -6.745498, -6.109509, -1.389960, -6.996812, -3.245629, -1.810225, -8.942722, -2.613001, -6.745498],   
    [ -3.258097, -3.258097, -3.258097, -1.648659, -2.564949, -3.258097, -3.258097, -3.258097, -3.258097, -3.258097, -1.466337, -3.258097, -3.258097, -3.258097, -3.258097, -3.258097],   
    [ -7.776954, -6.678342, -7.776954, -2.135047, -7.776954, -7.083807, -1.295377, -6.678342, -4.732432, -1.615747, -6.678342, -2.053369, -1.877057, -6.390660, -2.197225, -7.776954],   
    [ -4.025352, -4.025352, -4.025352, -2.926739, -4.025352, -4.025352, -1.317301, -4.025352, -2.079442, -2.415914, -4.025352, -4.025352, -1.317301, -4.025352, -4.025352, -4.025352],   
  ]

Pbulgedist = [-0.563112, -1.526062, -2.392811, -2.928284, -3.225183, -4.458385, -4.449946, -5.024377, -4.416886, -6.521020, -7.149628, -7.031845, -9.229070, -8.535923, -8.535923, -9.229070, -7.842776, -9.229070, -8.130458, -8.535923, -9.229070, -7.619632, -9.229070, -8.130458, -9.229070, -8.130458, -7.283160, -9.229070, -9.229070, -9.229070]
Pintloopdist = [ 
    [-1.406417, -2.902316, -3.650335, -4.933183, -5.920933, -6.818874, -6.965478, -8.118157, -8.118157, -8.811305, -8.523622, -9.216770, -8.523622, -9.216770, -9.216770, -8.523622, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917],
    [-3.392246, -2.299064, -3.410130, -5.105896, -4.933183, -6.148717, -7.964007, -7.830475, -8.523622, -7.964007, -8.523622, -6.774423, -8.523622, -9.216770, -8.523622, -8.300479, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917],
    [-3.827698, -3.794025, -2.579512, -3.874435, -5.997894, -3.567795, -7.607332, -7.019545, -7.607332, -8.811305, -8.523622, -9.216770, -8.811305, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -9.909917, -9.909917, -9.909917],
    [-4.353089, -4.797929, -3.798450, -3.337634, -4.803971, -4.562809, -6.691041, -6.965478, -6.081275, -5.849474, -9.216770, -8.300479, -9.216770, -9.216770, -8.523622, -8.811305, -9.216770, -8.523622, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917],
    [-5.114126, -5.455570, -5.105896, -4.307798, -4.318930, -4.750862, -4.968274, -7.512022, -8.300479, -7.425010, -7.712692, -9.216770, -9.216770, -9.216770, -8.300479, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-7.344967, -6.508719, -5.156327, -4.700431, -4.429278, -5.782782, -6.542621, -7.076703, -8.300479, -8.300479, -8.300479, -9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-7.201867, -7.712692, -6.354569, -6.577712, -5.866866, -5.553208, -7.425010, -6.731863, -7.964007, -8.811305, -7.712692, -8.300479, -8.523622, -8.523622, -8.523622, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-7.712692, -6.577712, -7.964007, -7.137328, -8.118157, -5.304747, -7.201867, -6.508719, -7.830475, -8.811305, -8.523622, -9.216770, -8.523622, -9.216770, -9.216770, -9.216770, -8.300479, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.811305, -8.300479, -7.712692, -8.300479, -8.523622, -6.731863, -7.201867, -7.712692, -7.830475, -8.300479, -9.216770, -8.523622, -8.811305, -8.523622, -8.300479, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.300479, -8.300479, -8.523622, -7.270860, -8.523622, -8.118157, -8.300479, -7.964007, -8.300479, -8.118157, -8.523622, -8.811305, -9.216770, -8.300479, -8.811305, -9.216770, -9.216770, -8.300479, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.118157, -7.964007, -9.216770, -6.542621, -7.830475, -8.811305, -8.523622, -8.811305, -7.607332, -8.300479, -8.523622, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -8.523622, -9.216770, -7.270860, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -8.811305, -7.964007, -8.118157, -8.118157, -7.964007, -7.270860, -6.965478, -7.712692, -9.216770, -8.811305, -9.216770, -9.216770, -8.300479, -9.216770, -9.216770, -9.216770, -8.811305, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.523622, -8.811305, -9.216770, -7.964007, -8.523622, -7.712692, -8.811305, -8.118157, -7.712692, -8.523622, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -8.811305, -8.300479, -7.830475, -8.523622, -8.523622, -9.216770, -9.216770, -9.216770, -8.811305, -8.523622, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -8.811305, -9.216770, -8.811305, -8.811305, -8.811305, -9.216770, -8.523622, -9.216770, -8.523622, -9.216770, -8.118157, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -8.811305, -9.216770, -8.811305, -9.216770, -8.811305, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -8.523622, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -8.811305, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -8.811305, -8.811305, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -8.811305, -8.811305, -8.811305, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.523622, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.216770, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.811305, -9.216770, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-8.523622, -9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.216770, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917],
    [-9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917, -9.909917]
  ]





In [87]:
pHL = [math.log(0.95)]


if num_HLMotifs > 0:
    for t in range(0, num_HLMotifs):
        pHL.append(math.log(0.05/num_HLMotifs))

print(pHL)
print(num_HLMotifs)

[-0.05129329438755058, -2.995732273553991]
1


In [88]:
pIL = []

if num_ILMotifs > 0:
    for t in range(0, num_ILMotifs):
        pIL.append(math.log(1/num_ILMotifs))
        
print(pIL)

[-0.6931471805599453, -0.6931471805599453]


In [89]:
bplist = ["AA", "AC", "AG", "AU", "CA", "CC", "CG", "CU", "GA", "GC", "GG", "GU", "UA", "UC", "UG", "UU"]
def bptonum(bp):
    charbp = bp
    for i in range(0, 16):
        if charbp == bplist[i]:
            return i


In [90]:
def covcomplete(i, j, covpairs):
    if covpairs == []:
        return 1
    else:
        pset = []
        for el in covpairs:
            if i <= el[0] <= j and i <= el[1] <= j:
                pset.append(1)
            elif el[1] < i:
                pset.append(1)
            elif el[0] > j:
                pset.append(1)
            elif el[0] < i and el[1] > j:
                pset.append(1)
            else:
                pset.append(0)
        prod = 1
        for el in pset:
            prod = prod*el
        return prod

def searchcov(i, j, covlist, covpairs):
    if covpairs == []:
        return 0
    else:
        sch = 0
        for el in covlist:
            if i <= el <= j:
                sch = 1
        return sch
    


In [91]:
def listtostr(arr):
    st = ""
    for el in arr:
        st+=el
    return st

In [92]:
pBlockNT = [math.log(0.9995), math.log(0.00025), math.log(0.00025)]
pBlockT = [math.log(0.998), math.log(0.002)]

In [93]:
#completes insertion symbol cyk matrix
def completeIn(RNAsequence, covpairs, lcov, rcov, covlist):
    seqlength = len(RNAsequence)
    maxlogpIn = []
    for j in range(0, seqlength):
        maxlogpInj = []
        maxlogpInj.append(-math.inf)
        for d in range(1, j+2):
            i = j-d+1
            if covcomplete(i, j, covpairs) == 0:
                maxlogpInj.append(-math.inf)
            else:
                if d == 1:
                    maxlogpInj.append(n_emissionsdict[RNAsequence[i]] + pIn[1])
                else:
                    maxlogpInd = -math.inf
                    if i in lcov:
                        maxlogpInj.append(-math.inf)
                    else:
                        psplit = maxlogpInj[d-1] + n_emissionsdict[RNAsequence[i]] + pIn[0]
                        if psplit > maxlogpInd:
                            maxlogpInd = psplit
                        maxlogpInj.append(maxlogpInd)
        maxlogpIn.append(maxlogpInj)
    return maxlogpIn


#completes HMM cyk matrix

def completeHMM(block, RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist):
    seqlength = len(RNAsequence)
    blocklength = len(block)
    maxlogpBlock = []
    for g in range(0, blocklength):
        maxlogpBlock.append([])
    for j in range(0, seqlength):
        maxlogpBlockj = []
        for g in range(0, blocklength):
            maxlogpBlockj.append([pBlockNT[1]*g + pBlockT[1]])
        for d in range(1, j+2):
            i = j-d+1
            if covcomplete(i, j, covpairs) == 0 or searchcov(i, j, covlist, covpairs) == 1:
                for g in range(0, blocklength):
                    maxlogpBlockj[g].append(-math.inf)
            else:
                if d == 1:
                    for g in range(0, blocklength):
                        if g == 0:
                            maxlogpBlockj[g].append(pBlockT[0] + letter_emissionsdict[block[-1]][RNAsequence[i]])
                        else:
                            maxlogpBlockd = pBlockNT[0] + letter_emissionsdict[block[-1-g]][RNAsequence[i]] + maxlogpBlockj[g-1][0]
                            psplit = pBlockNT[1] + maxlogpBlockj[g-1][1]
                            if psplit > maxlogpBlockd:
                                maxlogpBlockd = psplit
                            maxlogpBlockj[g].append(maxlogpBlockd)
                else:
                    for g in range(0, blocklength):
                        if g == 0:
                            maxlogpBlockj[g].append(-math.inf)
                        else:
                            if i in lcov:
                                maxlogpBlockd = -math.inf
                            else:
                                maxlogpBlockd = -math.inf
                                psplit = maxlogpBlockj[g-1][d-1] + letter_emissionsdict[block[-1-g]][RNAsequence[i]] + pBlockNT[0]
                                if psplit > maxlogpBlockd:
                                    maxlogpBlockd = psplit
                                if d == 2:
                                    pass
                                else:
                                    for s in range(0, d-2):
                                        psplit = letter_emissionsdict[block[-1-g]][RNAsequence[i]] + maxlogpIn[i+s+1][s+1] + maxlogpBlockj[g-1][d-s-2] + pBlockNT[2]
                                        if psplit > maxlogpBlockd:
                                            maxlogpBlockd = psplit
                                psplit = letter_emissionsdict[block[-1-g]][RNAsequence[i]] + maxlogpIn[j][d-1] + maxlogpBlockj[g-1][0] + pBlockNT[2]
                                if psplit > maxlogpBlockd:
                                    maxlogpBlockd = psplit
                            psplit = maxlogpBlockj[g-1][d] + pBlockNT[1]
                            if psplit > maxlogpBlockd:
                                maxlogpBlockd = psplit
                            maxlogpBlockj[g].append(maxlogpBlockd)
        for g in range(0, blocklength):
            maxlogpBlock[g].append(maxlogpBlockj[g])
    return maxlogpBlock


#completes length distribution cyk matrix
def completeHLBulge(RNAsequence, covpairs, lcov, rcov, covlist):
    seqlength = len(RNAsequence)
    maxlogpHLBulge = []
    for j in range(0, seqlength):
        maxlogpHLBulgej = []
        maxlogpHLBulgej.append(-math.inf)
        for d in range(1, j+2):
            i = j-d+1
            maxlogpHLBulged = -math.inf
            if covcomplete(i, j, covpairs) == 0:
                maxlogpHLBulged = -math.inf
            else:
                if d <= len(HLlendist):
                    if searchcov(i, j, covlist, covpairs) == 0:
                        psplit = pHL[0] + HLlendist[d-1]
                        for nlt in RNAsequence[i:j+1]:
                            psplit += hlsing_emissionsdict[nlt]
                        maxlogpHLBulged = psplit
            maxlogpHLBulgej.append(maxlogpHLBulged)
        maxlogpHLBulge.append(maxlogpHLBulgej)
    return maxlogpHLBulge

#completes length distribution cyk matrix
def completeHLExt(RNAsequence, covpairs, lcov, rcov, covlist):
    seqlength = len(RNAsequence)
    maxlogpHLExt = []
    for j in range(0, seqlength):
        maxlogpHLExtj = []
        maxlogpHLExtj.append(pHLExt[1])
        for d in range(1, j+2):
            i = j-d+1
            maxlogpHLExtd = -math.inf
            if covcomplete(i, j, covpairs) == 0:
                maxlogpHLExtd = -math.inf
            else:
                if i in lcov:
                    maxlogpHLExtd = -math.inf
                else:
                    maxlogpHLExtd = pHLExt[0] + n_emissionsdict[RNAsequence[i]] + maxlogpHLExtj[d-1]
            maxlogpHLExtj.append(maxlogpHLExtd)
        maxlogpHLExt.append(maxlogpHLExtj)
    return maxlogpHLExt
                    
                        
                    
            
def completeBlank(RNAsequence):
    seqlength = len(RNAsequence)
    blank = []
    for j in range(0, seqlength):
        blankj = []
        for d in range(1, j+2):
            blankj.append(0)
        blank.append(blankj)
    return blank


#completes length distribution cyk matrix for bulge internal loop
def completeILBulge(RNAsequence, covpairs, lcov, rcov, covlist):
    seqlength = len(RNAsequence)
    maxlogpILBulge = []
    for j in range(0, seqlength):
        maxlogpILBulgej = []
        maxlogpILBulgej.append(ILlendist[0])
        for d in range(1, j+2):
            i = j-d+1
            maxlogpILBulged = -math.inf
            if covcomplete(i, j, covpairs) == 0:
                maxlogpILBulged = -math.inf
            else:
                if d <= len(ILlendist)-1:
                    if searchcov(i, j, covlist, covpairs) == 0:
                        psplit = ILlendist[d]
                        for nlt in RNAsequence[i:j+1]:
                            psplit += qsing_emissionsdict[nlt]
                        maxlogpILBulged = psplit
            maxlogpILBulgej.append(maxlogpILBulged)
        maxlogpILBulge.append(maxlogpILBulgej)
    return maxlogpILBulge


#completes length distribution cyk matrix
def completeILExt(RNAsequence, covpairs, lcov, rcov, covlist):
    seqlength = len(RNAsequence)
    maxlogpILExt = []
    for j in range(0, seqlength):
        maxlogpILExtj = []
        maxlogpILExtj.append(pILExt[1])
        for d in range(1, j+2):
            i = j-d+1
            maxlogpILExtd = -math.inf
            if covcomplete(i, j, covpairs) == 0:
                maxlogpILExtd = -math.inf
            else:
                if i in lcov:
                    maxlogpILExtd = -math.inf
                else:
                    maxlogpILExtd = pILExt[0] + n_emissionsdict[RNAsequence[i]] + maxlogpILExtj[d-1]
            maxlogpILExtj.append(maxlogpILExtd)
        maxlogpILExt.append(maxlogpILExtj)
    return maxlogpILExt

In [94]:
#BP key:
#IUPAC - IUPAC: is normally interpreted (each symbol independently)
#C-G/G-C: 
#A-U/U-A:
#Any bp: 






In [95]:
watson_crick = [["A", "U"], ["C", "G"], ["AU", "AU"], ["CG", "CG"], ["WC", "WC"]]
wobble = [["G", "U"], ["WB", "WB"]]

def pDeleteBp(l, r):
    if [l, r] in watson_crick or [r, l] in watson_crick:
        return math.log(0.001)
    if [l, r] in wobble or [r, l] in wobble:
        return math.log(0.05)
    else:
        return math.log(0.2)
        



In [96]:
letter_IUPAC = {"A" : ["A"], "G" : ["G"], "C" : ["C"], "U" : ["U"], "W" : ["A", "U"], "S" : ["G", "C"], "M" : ["A", "C"], "K" : ["G", "U"], "R" : ["A", "G"], "Y" : ["C", "U"], "B" : ["C", "G", "U"], "D" : ["A", "G", "U"], "H":["A", "C", "U"], "V":["A", "C", "G"], "N":["A", "G", "C", "U"]}


In [97]:
pBp = 1
totalBp = 16

def pBpEmit(observed, true):
    l = observed[0]
    r = observed[1]
    t_l = true[0]
    t_r = true[1]
    if t_l in letter_IUPAC and t_r in letter_IUPAC:
        total_comb = len(letter_IUPAC[t_l])*len(letter_IUPAC[t_r])
        if l in letter_IUPAC[t_l] and r in letter_IUPAC[t_r]:
            return math.log(pBp/total_comb)
        else:
            return math.log((1-pBp)/(totalBp - total_comb))        
    else:
        if true == ["AU", "AU"]:
            if observed == ["A", "U"] or observed == ["U", "A"]:
                return math.log(pBp/2)
            else:
                return math.log(pBp/(totalBp - 2))
        if true == ["CG", "CG"]:
            if observed == ["C", "G"] or observed == ["C", "G"]:
                return math.log(pBp/2)
            else:
                return math.log(pBp/(totalBp - 2))
        if true == ["WC", "WC"]:
            if observed == ["A", "U"] or observed == ["U", "A"] or observed == ["C", "G"] or observed == ["C", "G"]:
                return math.log(pBp/4)
            else:
                return math.log(pBp/(totalBp - 4))
        if true == ["WB", "WB"]:
            if observed == ["G", "U"] or observed == ["U", "G"]:
                return math.log(pBp/2)
            else:
                return math.log(pBp/(totalBp - 2))

In [98]:
def parseHMM(RNAsequence, maxlogpIn, matrix, parsenew, subseqnew, block, j, d, t, typ, parse, rebuild, reduc):
    i = j - d + 1
    if block == "Bulge":
        if parse[t] == typ+"."+str(len("Bulge")):
            prlt = ["n"]
            subseqlt = [[i, 1]]
            for s in range(t+1, t+d):
                prlt.append("n")
                subseqlt.append([s-t+i, 1])                   
            for s in range(i, j+1):
                rebuild[s] = RNAsequence[s]
                reduc[s] = "x"
            parsenew.append(prlt)
            subseqnew.append(subseqlt)
        
    
    else:
    
        for g in range(0, len(block)):
            if parse[t] == typ+"."+str(g+1):
                if g == 0:
                    if d == 0:
                        pass
                    if d == 1:
                        parsenew.append([block[-1].lower()])
                        subseqnew.append([[j, d]])
                        rebuild[i] = RNAsequence[i]
                        reduc[i] = block[-1].lower()
                else:
                    if d == 0:
                        parsenew.append([typ+"."+str(g)])
                        subseqnew.append([[j, d]])
                    elif d == 1:
                        if abs(matrix[g][j][d] - (matrix[g-1][j][d-1] + letter_emissionsdict[block[-1-g]][RNAsequence[i]] + pBlockNT[0])) < 10e-13:
                            parsenew.append([block[-1-g].lower(), typ+"."+str(g)])
                            subseqnew.append([[i, 1], [j, d-1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = block[-1-g].lower()
                        elif abs(matrix[g][j][d] - (matrix[g-1][j][d] + pBlockNT[1])) < 10e-13:
                            parsenew.append([typ+"."+str(g)])
                            subseqnew.append([[j, d]])

                    else:
                        if abs(matrix[g][j][d] - (matrix[g-1][j][d-1] + letter_emissionsdict[block[-1-g]][RNAsequence[i]] + pBlockNT[0])) < 10e-13:
                            parsenew.append([block[-1-g].lower(), typ+"."+str(g)])
                            subseqnew.append([[i, 1], [j, d-1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = block[-1-g].lower()
                        elif abs(matrix[g][j][d] - (matrix[g-1][j][d] + pBlockNT[1])) < 10e-13:
                            parsenew.append([typ+"."+str(g)])
                            subseqnew.append([[j, d]])
                        for s in range(1, d):
                            if abs(matrix[g][j][d] - (letter_emissionsdict[block[-1-g]][RNAsequence[i]] + maxlogpIn[i+s][s] + matrix[g-1][j][d-s-1] + pBlockNT[2])) < 10e-13:
                                parsenew.append([block[-1-g].lower(), "In", typ+"."+str(g)])
                                subseqnew.append([[i, 1], [i+s, s], [j, d-s-1]])
                                rebuild[i] = RNAsequence[i]
                                reduc[i] = block[-1-g].lower()

In [99]:
def parsesequence(RNAsequence, covpairs, restrictbp):
    Anum = RNAsequence.count("A")
    Gnum = RNAsequence.count("G")
    Cnum = RNAsequence.count("C")
    Unum = RNAsequence.count("U")
    Nnum = Anum + Gnum + Cnum + Unum
    if RNAsequence == "":
        return None
    elif Nnum != len(RNAsequence):
        return None    
    else:
        if covpairs == []:
            covlist = []
            lcov = []
            rcov = []
        else:
            lcov = []
            rcov = []
            covlist = []
            for el in covpairs:
                lcov.append(el[0])
                rcov.append(el[1])
                covlist.append(el[0])
                covlist.append(el[1])
        n = len(RNAsequence)
        maxlogpS = []
        maxlogpL = []
        maxlogpF = []
        maxlogpP = []
        maxlogpHL = []
        if num_ILMotifs > 0:
            maxlogpIL = []
        maxlogpM = []
        maxlogpR = []
        maxlogpM1 = []
        #insert is prefilled
        maxlogpIn = completeIn(RNAsequence, covpairs, lcov, rcov, covlist)
        #hairpin loop bulge distribution is prefilled
        maxlogpHLBulge = completeHLBulge(RNAsequence, covpairs, lcov, rcov, covlist)
        #contiguous blocks are prefilled in the hairpin loop
        maxlogpHLloopBlocks = []
        if num_HLMotifs > 0:
            for t in range(0, num_HLMotifs):
                if HLPacket[t][0] == "Bulge":
                    maxlogpHLloopBlocks.append([maxlogpHLBulge])
                else: 
                    maxlogpHLloopBlocks.append(completeHMM(HLPacket[t][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
        #contiguous hairpin stem loop blocks are prefilled
        maxlogpHLleftStem = []
        maxlogpHLrightStem = []
        if num_HLMotifs > 0:
            for t in range(0, num_HLMotifs):
                lpack = []
                rpack = []
                for h in range(0, len(HLPacket[t][1][0])):
                    if HLPacket[t][1][0][h][1] == 1:
                        lpack.append("bp")
                    else:
                        L = len(HLPacket[t][1][0][h][0])
                        if L == 0:
                            lpack.append(completeBlank(RNAsequence))
                        else:
                            lpack.append(completeHMM(HLPacket[t][1][0][h][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                for h in range(0, len(HLPacket[t][1][1])):
                    if HLPacket[t][1][1][h][1] == 1:
                        rpack.append("bp")
                    else:
                        L = len(HLPacket[t][1][1][h][0])
                        if L == 0:
                            rpack.append(completeBlank(RNAsequence))
                        else:
                            rpack.append(completeHMM(HLPacket[t][1][1][h][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                maxlogpHLleftStem.append(lpack)
                maxlogpHLrightStem.append(rpack)
        #intermediaries in Motifk --> Int1 --> Int2 --> ... --> Loop
        maxlogpHLmed = []
        if num_HLMotifs > 0:
            for t in range(0, num_HLMotifs):
                tMotif = []
                for g in range(0, num_HLStem[t]):
                    tMotif.append([])
                maxlogpHLmed.append(tMotif)
        #Insertions at the end of the loop sequence portion of the motif are prefilled
        maxlogpHLExt = completeHLExt(RNAsequence, covpairs, lcov, rcov, covlist) 

       
        #internal loop bulge distribution is prefilled
        maxlogpILBulge = completeILBulge(RNAsequence, covpairs, lcov, rcov, covlist)
        
        #internal loop left bulge and right contiguous blocks are prefilled
        maxlogpILLeft = []
        maxlogpILRight = []
        if num_ILMotifs > 0:
            for t in range(0, num_ILMotifs):
                if ILPacket[t][1][0] == "Bulge":
                    maxlogpILLeft.append([maxlogpILBulge])
                else:
                    maxlogpILLeft.append(completeHMM(ILPacket[t][1][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                if ILPacket[t][1][1] == "Bulge":
                    maxlogpILRight.append([maxlogpILBulge])
                else:
                    maxlogpILRight.append(completeHMM(ILPacket[t][1][1], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                    
        
        #contiguous stem blocks are prefilled
        #inside
        maxlogpILInLeftStem = []
        maxlogpILInRightStem = []
        if num_ILMotifs > 0:
            for t in range(0, num_ILMotifs):
                lpack = []
                rpack = []
                for h in range(0, len(ILPacket[t][0][0])):
                    if ILPacket[t][0][0][h][1] == 1:
                        lpack.append("bp")
                    else:
                        L = len(ILPacket[t][0][0][h][0])
                        if L == 0:
                            lpack.append(completeBlank(RNAsequence))
                        else:
                            lpack.append(completeHMM(ILPacket[t][0][0][h][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                for h in range(0, len(ILPacket[t][0][1])):
                    if ILPacket[t][0][1][h][1] == 1:
                        rpack.append("bp")
                    else:
                        L = len(ILPacket[t][0][1][h][0])
                        if L == 0:
                            rpack.append(completeBlank(RNAsequence))
                        else:
                            rpack.append(completeHMM(ILPacket[t][0][1][h][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                maxlogpILInLeftStem.append(lpack)
                maxlogpILInRightStem.append(rpack)
        
                
        #outside
        maxlogpILOutLeftStem = []
        maxlogpILOutRightStem = []
        if num_ILMotifs > 0:
            for t in range(0, num_ILMotifs):
                lpack = []
                rpack = []
                for h in range(0, len(ILPacket[t][2][0])):
                    if ILPacket[t][2][0][h][1] == 1:
                        lpack.append("bp")
                    else:
                        L = len(ILPacket[t][2][0][h][0])
                        if L == 0:
                            lpack.append(completeBlank(RNAsequence))
                        else:
                            lpack.append(completeHMM(ILPacket[t][2][0][h][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                for h in range(0, len(ILPacket[t][2][1])):
                    if ILPacket[t][2][1][h][1] == 1:
                        rpack.append("bp")
                    else:
                        L = len(ILPacket[t][2][1][h][0])
                        if L == 0:
                            rpack.append(completeBlank(RNAsequence))
                        else:
                            rpack.append(completeHMM(ILPacket[t][2][1][h][0], RNAsequence, maxlogpIn, covpairs, lcov, rcov, covlist))
                maxlogpILOutLeftStem.append(lpack)
                maxlogpILOutRightStem.append(rpack)
        
        #intermediaries in Motifk --> Int1 --> Int2 --> ... --> Loop
        #inside
        maxlogpILInmed = []
        if num_ILMotifs > 0:
            for t in range(0, num_ILMotifs):
                tMotif = []
                for g in range(0, num_ILInStem[t]):
                    tMotif.append([])
                maxlogpILInmed.append(tMotif)
        
        #Loop --> lBulge Out1 rBulge; Out1 --> Out2 --> ... --> L
        #outside
        maxlogpILOutmed = []
        if num_ILMotifs > 0:
            for t in range(0, num_ILMotifs):
                tMotif = []
                for g in range(0, num_ILOutStem[t]):
                    tMotif.append([])
                maxlogpILOutmed.append(tMotif)
        
        #Insertions at the end of the loop sequence portion of the motif are prefilled
        maxlogpILExt = completeILExt(RNAsequence, covpairs, lcov, rcov, covlist)
        
        for j in range(0, n):
            maxlogpSj = []
            maxlogpLj = []
            maxlogpFj = []
            maxlogpPj = []
            maxlogpHLj = []
            if num_ILMotifs > 0:
                maxlogpILj = []
            maxlogpMj = []
            maxlogpRj = []
            maxlogpM1j = []
            
            
            maxlogpHLmedj = []
            if num_HLMotifs > 0:
                for t in range(0, num_HLMotifs):
                    tMotifj = []
                    for g in range(0, num_HLStem[t]):
                        tMotifj.append([])
                    maxlogpHLmedj.append(tMotifj)
            
            maxlogpILInmedj = []
            if num_ILMotifs > 0:
                for t in range(0, num_ILMotifs):
                    tMotifj = []
                    for g in range(0, num_ILInStem[t]):
                        tMotifj.append([])
                    maxlogpILInmedj.append(tMotifj)
            
            maxlogpILOutmedj = []
            if num_ILMotifs > 0:
                for t in range(0, num_ILMotifs):
                    tMotifj = []
                    for g in range(0, num_ILOutStem[t]):
                        tMotifj.append([])
                    maxlogpILOutmedj.append(tMotifj)
            
            
            maxlogpSj.append(pS[2])
            maxlogpLj.append(-math.inf)
            maxlogpFj.append(-math.inf)
            maxlogpMj.append(-math.inf)
            maxlogpRj.append(-math.inf)
            maxlogpM1j.append(-math.inf)
            #edit after internal loop
            maxlogpPj.append(-math.inf)
            
            if num_HLMotifs > 0:
                for t in range(0, num_HLMotifs):
                    for g in range(0, num_HLStem[t]):
                        if g == 0:
                            maxlogpHLmedj[t][g].append(maxlogpHLExt[j][0] + maxlogpHLloopBlocks[t][-1][j][0])
                        else:
                            if g >= bpcut_HLStem[t]:
                                maxlogpHLmedj[t][g].append(-math.inf)
                            else:
                                 maxlogpHLmedj[t][g].append(maxlogpHLleftStem[t][-g-1][-1][j][0] + maxlogpHLmedj[t][g-1][0] + maxlogpHLrightStem[t][g][-1][j][0])

            tot = [-math.inf]
            if num_HLMotifs > 0:
                for t in range(0, num_HLMotifs):
                    tot.append(maxlogpHLmedj[t][-1][0] + pHL[t+1])
            maxlogpHLj.append(max(tot))
            
            
            
            if num_ILMotifs > 0:
                for t in range(0, num_ILMotifs):
                    for g in range(0, num_ILInStem[t]):
                        maxlogpILInmedj[t][g].append(-math.inf)
                        
            if num_ILMotifs > 0:
                for t in range(0, num_ILMotifs):
                    for g in range(0, num_ILOutStem[t]):
                        maxlogpILOutmedj[t][g].append(-math.inf)
            if num_ILMotifs > 0:            
                maxlogpILj.append(-math.inf)

                                
            for d in range(1, j+2):
                i = j-d+1
                if covcomplete(i, j, covpairs) == 0:
                    maxlogpSj.append(-math.inf)
                    maxlogpLj.append(-math.inf)
                    maxlogpFj.append(-math.inf)
                    maxlogpMj.append(-math.inf)
                    maxlogpRj.append(-math.inf)
                    maxlogpM1j.append(-math.inf)
                    maxlogpPj.append(-math.inf)
                    maxlogpHLj.append(-math.inf)
                    if num_ILMotifs > 0:
                        maxlogpILj.append(-math.inf)
                    if num_HLMotifs > 0:
                        for t in range(0, num_HLMotifs):
                            for g in range(0, num_HLStem[t]):
                                maxlogpHLmedj[t][g].append(-math.inf)
                    
                    if num_ILMotifs > 0:
                        for t in range(0, num_ILMotifs):
                            for g in range(0, num_ILInStem[t]):
                                maxlogpILInmedj[t][g].append(-math.inf)

                    if num_ILMotifs > 0:
                        for t in range(0, num_ILMotifs):
                            for g in range(0, num_ILOutStem[t]):
                                maxlogpILOutmedj[t][g].append(-math.inf)
                
                else:
                    if d == 1:
                        #S
                        maxlogpSd = pS[0] + n_emissionsdict[RNAsequence[i]] + pS[2]
                        maxlogpSj.append(maxlogpSd)
                        #L
                        maxlogpLd = -math.inf
                        maxlogpLj.append(maxlogpLd)
                        #F
                        maxlogpFd = -math.inf
                        maxlogpFj.append(maxlogpFd)
                        #M1
                        maxlogpM1d = -math.inf
                        maxlogpM1j.append(maxlogpM1d)
                        #R
                        maxlogpRd = -math.inf
                        maxlogpRj.append(maxlogpRd)
                        #M
                        maxlogpMd = -math.inf
                        maxlogpMj.append(maxlogpMd)
                        #Intermediary HL Stem Terminals
                        if num_HLMotifs > 0:
                            for t in range(0, num_HLMotifs):
                                for g in range(0, num_HLStem[t]):
                                    if g == 0:
                                        opt1 = maxlogpHLExt[j][0] + maxlogpHLloopBlocks[t][-1][j][d]
                                        opt2 = maxlogpHLExt[j][d] + maxlogpHLloopBlocks[t][-1][j][0]
                                        maxlogpHLmedj[t][g].append(max(opt1, opt2))
                                    else:
                                        if g >= bpcut_HLStem[t]:
                                            maxlogpHLmedj[t][g].append(-math.inf)
                                        else:
                                            opt1 = maxlogpHLleftStem[t][-g-1][-1][j][d] + maxlogpHLmedj[t][g-1][0] + maxlogpHLrightStem[t][g][-1][j][0]
                                            opt2 = maxlogpHLleftStem[t][-g-1][-1][j][0] + maxlogpHLmedj[t][g-1][0] + maxlogpHLrightStem[t][g][-1][j][d]
                                            opt3 = maxlogpHLleftStem[t][-g-1][-1][j][0] + maxlogpHLmedj[t][g-1][d] + maxlogpHLrightStem[t][g][-1][j][0]
                                            maxlogpHLmedj[t][g].append(max(opt1, opt2, opt3))
                        #P (needs to include HL fix later)
                        maxlogpPd = -math.inf
                        maxlogpPj.append(maxlogpPd)
                        
                        #HL
                        totset = [-math.inf]
                        if num_HLMotifs > 0:
                            for t in range(0, num_HLMotifs):
                                totset.append(maxlogpHLmedj[t][-1][d] + pHL[t+1])
                        maxlogpHLj.append(max(totset))
                        
                        #IL
                        if num_ILMotifs > 0:
                            maxlogpILj.append(-math.inf)
                        
                        #Intermediary IL In Stem Terminals
                        if num_ILMotifs > 0:
                            for t in range(0, num_ILMotifs):
                                for g in range(0, num_ILInStem[t]):
                                    maxlogpILInmedj[t][g].append(-math.inf)
                        
                        #Intermediary IL Out Stem Terminals
                        if num_ILMotifs > 0:
                            for t in range(0, num_ILMotifs):
                                for g in range(0, num_ILOutStem[t]):
                                    maxlogpILOutmedj[t][g].append(-math.inf)

                        
                    else:
                        maxlogpSd = -math.inf
                        maxlogpLd = -math.inf
                        maxlogpFd = -math.inf
                        maxlogpPd = -math.inf
                        maxlogpHLd = -math.inf
                        maxlogpMd = -math.inf
                        maxlogpRd = -math.inf
                        maxlogpM1d = -math.inf
                        
                        #F
                        if [i, j] in restrictbp:
                            maxlogpFj.append(-math.inf)
                        else:
                            if [i, j] not in covpairs and i in lcov and j in rcov:
                                maxlogpFd = -math.inf
                            elif [i, j] not in covpairs and i not in lcov and j in rcov:
                                maxlogpFd = -math.inf
                            elif [i, j] not in covpairs and i in lcov and j not in rcov:
                                maxlogpFd = -math.inf
                            else:
                                if i == 0 or j == n-1:
                                    maxlogpFd = -math.inf
                                else:
                                    key = bptonum(RNAsequence[i-1] + RNAsequence[j+1])
                                    key1 = bptonum(RNAsequence[i] + RNAsequence[j])
                                    psplit = maxlogpF[j-1][d-2] + stackedbpF1[key][key1] + pF[0]
                                    if psplit > maxlogpFd:
                                        maxlogpFd = psplit
                                    psplit = maxlogpP[j-1][d-2] + stackedbpF2[key][key1] + pF[1]
                                    if psplit > maxlogpFd:
                                        maxlogpFd = psplit              
                            maxlogpFj.append(maxlogpFd)
                        #L
                        if [i, j] in restrictbp:
                            maxlogpLj.append(-math.inf)
                        else:
                            if [i, j] not in covpairs and i in lcov and j in rcov:
                                maxlogpLd = -math.inf
                            elif [i, j] not in covpairs and i not in lcov and j in rcov:
                                maxlogpLd = -math.inf
                            elif [i, j] not in covpairs and i in lcov and j not in rcov:
                                maxlogpLd = -math.inf
                            else:
                                if d == 2:            
                                    psplit = maxlogpPj[0] + pbpL2[bptonum(RNAsequence[i] + RNAsequence[j])] + pL[1]
                                    if psplit > maxlogpLd:
                                        maxlogpLd = psplit
                                else:
                                    psplit = maxlogpF[j-1][d-2] + pbpL1[bptonum(RNAsequence[i] + RNAsequence[j])] + pL[0]
                                    
                                    if psplit > maxlogpLd:
                                        maxlogpLd = psplit
                                    psplit = maxlogpP[j-1][d-2] + pbpL2[bptonum(RNAsequence[i] + RNAsequence[j])] + pL[1]
                                    if psplit > maxlogpLd:
                                        maxlogpLd = psplit
                            maxlogpLj.append(maxlogpLd)
                        #S
                        if i in lcov:
                            maxlogpSd = -math.inf
                        else:
                            psplit = maxlogpSj[d-1] + n_emissionsdict[RNAsequence[i]] + pS[0]
                            if psplit > maxlogpSd:
                                maxlogpSd = psplit
                        for s in range(0, d-1):
                            psplit = maxlogpL[i+s][s+1] + maxlogpSj[d-1-s] + pS[1]   
                            if psplit > maxlogpSd:
                                maxlogpSd = psplit
                        psplit = maxlogpLd + maxlogpSj[0] + pS[1]
                        if psplit > maxlogpSd:
                            maxlogpSd = psplit
                        maxlogpSj.append(maxlogpSd)
                        #M1
                        if i in lcov:
                            maxlogpM1d = -math.inf
                        else:
                            psplit = maxlogpM1j[d-1] + n_emissionsdict[RNAsequence[i]] + pM1[0]
                            if psplit > maxlogpM1d:
                                maxlogpM1d = psplit
                        psplit = maxlogpLd + pM1[1]
                        if psplit > maxlogpM1d:
                            maxlogpM1d = psplit
                        maxlogpM1j.append(maxlogpM1d)
                        #R
                        if j in rcov:
                            maxlogpRd = -math.inf
                        else:
                            psplit = maxlogpR[j-1][d-1] + n_emissionsdict[RNAsequence[j]] + pR[0]
                            if psplit > maxlogpRd:
                                maxlogpRd = psplit
                        psplit = maxlogpM1d + pR[1]
                        if psplit > maxlogpRd:
                            maxlogpRd = psplit
                        maxlogpRj.append(maxlogpRd)
                        #M
                        for s in range(0, d-1):
                            psplit = maxlogpM1[i+s][s+1] + maxlogpMj[d-1-s] + pM[0]
                            if psplit > maxlogpMd:
                                maxlogpMd = psplit
                        psplit = maxlogpRd + pM[1]
                        if psplit > maxlogpMd:
                            maxlogpMd = psplit
                        maxlogpMj.append(maxlogpMd)
                        
                        #Intermediary HL Stem Loop Terminals
                        if num_HLMotifs > 0:
                            for t in range(0, num_HLMotifs):
                                for g in range(0, num_HLStem[t]):
                                    maxlogpHLmedd = -math.inf
                                    if g == 0:
                                        for s in range(0, d+1):
                                            psplit = maxlogpHLExt[j][d-s] + maxlogpHLloopBlocks[t][-1][i+s-1][s]
                                            if psplit > maxlogpHLmedd:
                                                maxlogpHLmedd = psplit
                                        
                                                                                
                                    else:
                                        if HLPacket[t][1][1][g-1][1] == 1:
                                            #base pair
                                            if [i, j] in restrictbp:
                                                pass
                                            else:
                                                if [i, j] not in covpairs and i in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i not in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i in lcov and j not in rcov:
                                                    pass
                                                else:
                                                    pThrough = math.log(1 - math.exp(pDeleteBp(HLPacket[t][1][0][-g][0], HLPacket[t][1][1][g-1][0])))
                                                    psplit = maxlogpHLmed[t][g-1][j-1][d-2] + pBpEmit([RNAsequence[i], RNAsequence[j]], [HLPacket[t][1][0][-g][0], HLPacket[t][1][1][g-1][0]]) + pThrough
                                                    if psplit > maxlogpHLmedd:
                                                        maxlogpHLmedd = psplit
                                            psplit = pDeleteBp(HLPacket[t][1][0][-g][0], HLPacket[t][1][1][g-1][0]) + maxlogpHLmedj[t][g-1][d]
                                            if psplit > maxlogpHLmedd:
                                                maxlogpHLmedd = psplit
                                        else:
                                            for s in range(0, d+1):
                                                psplit = maxlogpHLleftStem[t][-g][-1][i+s-1][s] + maxlogpHLmedj[t][g-1][d-s] + maxlogpHLrightStem[t][g-1][-1][j][0]
                                                if psplit > maxlogpHLmedd:
                                                    maxlogpHLmedd = psplit
                                            for x in range(0, d+1):
                                                for y in range(1, d-x+1):
                                                    psplit = maxlogpHLleftStem[t][-g][-1][i+x-1][x] + maxlogpHLmed[t][g-1][j-y][d-x-y] + maxlogpHLrightStem[t][g-1][-1][j][y]
                                                    if psplit > maxlogpHLmedd:
                                                        maxlogpHLmedd = psplit
                                    maxlogpHLmedj[t][g].append(maxlogpHLmedd)
                                    
                                                            
                        #HL
                        totalset = [maxlogpHLBulge[j][d]]
                        if num_HLMotifs > 0:
                            for t in range(0, num_HLMotifs):
                                totalset.append(maxlogpHLmedj[t][-1][d] + pHL[t+1])
                        maxlogpHLj.append(max(totalset))
                        
                        #Intermediary IL Out Stem Terminals
                        if num_ILMotifs > 0:
                            for t in range(0, num_ILMotifs):
                                for g in range(0, num_ILOutStem[t]):
                                    maxlogpILOutd = -math.inf
                                    if g == 0:
                                        #base pair when returning to L
                                        if ILPacket[t][2][1][g][1] == 1:
                                            if [i, j] in restrictbp:
                                                pass
                                            else:
                                                if [i, j] not in covpairs and i in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i not in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i in lcov and j not in rcov:
                                                    pass
                                                else:
                                                    pThrough = math.log((1 - math.exp(pDeleteBp(ILPacket[t][2][0][-1-g][0], ILPacket[t][2][1][g][0]))))
                                                    psplit = maxlogpL[j-1][d-2] + pBpEmit([RNAsequence[i], RNAsequence[j]], [ILPacket[t][2][0][-1-g][0], ILPacket[t][2][1][g][0]]) + pThrough
                                                    if psplit > maxlogpILOutd:
                                                        maxlogpILOutd = psplit
                                            psplit = pDeleteBp(ILPacket[t][2][0][-1-g][0], ILPacket[t][2][1][g][0]) + maxlogpLj[d]
                                            if psplit > maxlogpILOutd:
                                                maxlogpILOutd = psplit
                                            
                                        else:
                                            if i in lcov:
                                                pass
                                            else:
                                                for s in range(0, d+1):
                                                    psplit = maxlogpILOutLeftStem[t][-g-1][-1][i+s-1][s] + maxlogpLj[d-s] + maxlogpILOutRightStem[t][g][-1][j][0]
                                                    if psplit > maxlogpILOutd:
                                                        maxlogpILOutd = psplit
                                                for x in range(0, d+1):
                                                    for y in range(1, d-x+1):
                                                        psplit = maxlogpILOutLeftStem[t][-g-1][-1][i+x-1][x] + maxlogpL[j-y][d-x-y] + maxlogpILOutRightStem[t][g][-1][j][y]
                                                        if psplit > maxlogpILOutd:
                                                            maxlogpILOutd = psplit
                                        
                                    else:
                                        if ILPacket[t][2][1][g][1] == 1:
                                            if [i, j] in restrictbp:
                                                pass
                                            else:
                                                if [i, j] not in covpairs and i in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i not in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i in lcov and j not in rcov:
                                                    pass
                                                else:
                                                    pThrough = math.log(1 - math.exp(pDeleteBp(ILPacket[t][2][0][-1-g][0], ILPacket[t][2][1][g][0])))
                                                    psplit = maxlogpILOutmed[t][g-1][j-1][d-2] + pBpEmit([RNAsequence[i], RNAsequence[j]], [ILPacket[t][2][0][-1-g][0], ILPacket[t][2][1][g][0]]) + pThrough
                                                    if psplit > maxlogpILOutd:
                                                        maxlogpILOutd = psplit
                                                    psplit+=stackedbpF2[key][key1]
                                            psplit = pDeleteBp(ILPacket[t][2][0][-1-g][0], ILPacket[t][2][1][g][0]) + maxlogpILOutmedj[t][g-1][d]
                                            if psplit > maxlogpILOutd:
                                                maxlogpILOutd = psplit
                                            
                                        else:
                                            for s in range(0, d+1):                                                
                                                psplit = maxlogpILOutLeftStem[t][-g-1][-1][i+s-1][s] + maxlogpILOutmedj[t][g-1][d-s] + maxlogpILOutRightStem[t][g][-1][j][0]
                                                
                                                
                                                if psplit > maxlogpILOutd:
                                                    maxlogpILOutd = psplit
                                            for x in range(0, d+1):
                                                for y in range(1, d-x+1):
                                                    psplit = maxlogpILOutLeftStem[t][-g-1][-1][i+x-1][x] + maxlogpILOutmed[t][g-1][j-y][d-x-y] + maxlogpILOutRightStem[t][g][-1][j][y]
                                                    if psplit > maxlogpILOutd:
                                                        maxlogpILOutd = psplit
                                    
                                    maxlogpILOutmedj[t][g].append(maxlogpILOutd)
                        #Intermediary IL In Stem Terminals
                        
                        if num_ILMotifs > 0:
                            for t in range(0, num_ILMotifs):
                                for g in range(0, num_ILInStem[t]):
                                    maxlogpILInd = -math.inf
                                    if g == 0:
                                        if i in lcov:
                                            pass
                                        else:
                                            if num_ILOutStem[t] == 0:
                                                for s in range(0, d+1):
                                                    psplit = maxlogpILLeft[t][-1][i+s-1][s] + maxlogpLj[d-s] + maxlogpILRight[t][-1][j][0]
                                                    if psplit > maxlogpILInd:
                                                        maxlogpILInd = psplit

                                                for x in range(0, d+1):
                                                    for y in range(1, d-x+1):
                                                        psplit = maxlogpILLeft[t][-1][i+x-1][x] + maxlogpL[j-y][d-x-y] + maxlogpILRight[t][-1][j][y]
                                                        if psplit > maxlogpILInd:
                                                            maxlogpILInd = psplit

                                            else:
                                                for s in range(0, d+1):
                                                    psplit = maxlogpILLeft[t][-1][i+s-1][s] + maxlogpILOutmedj[t][-1][d-s] + maxlogpILRight[t][-1][j][0]
                                                    if psplit > maxlogpILInd:
                                                        maxlogpILInd = psplit
                                                for x in range(0, d+1):
                                                    for y in range(1, d-x+1):
                                                        psplit = maxlogpILLeft[t][-1][i+x-1][x] + maxlogpILOutmed[t][-1][j-y][d-x-y] + maxlogpILRight[t][-1][j][y]
                                                        if psplit > maxlogpILInd:
                                                            maxlogpILInd = psplit

                                    else:
                                        if ILPacket[t][0][1][g-1][1] == 1:
                                            if [i, j] in restrictbp:
                                                pass
                                            else:
                                                if [i, j] not in covpairs and i in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i not in lcov and j in rcov:
                                                    pass
                                                elif [i, j] not in covpairs and i in lcov and j not in rcov:
                                                    pass
                                                else:
                                                    pThrough = math.log(1 - math.exp(pDeleteBp(ILPacket[t][0][0][-g][0], ILPacket[t][0][1][g-1][0])))
                                                    psplit = maxlogpILInmed[t][g-1][j-1][d-2] + pThrough + pBpEmit([RNAsequence[i], RNAsequence[j]], [ILPacket[t][0][0][-g][0], ILPacket[t][0][1][g-1][0]])
                                                    if psplit > maxlogpILInd:
                                                        maxlogpILInd = psplit
                                            psplit = pDeleteBp(ILPacket[t][0][0][-g][0], ILPacket[t][0][1][g-1][0]) + maxlogpILInmedj[t][g-1][d]
                                            if psplit > maxlogpILInd:
                                                maxlogpILInd = psplit
                                        else:                                            
                                            for s in range(0, d+1):
                                                psplit = maxlogpILInLeftStem[t][-g][-1][i+s-1][s] + maxlogpILInmedj[t][g-1][d-s] + maxlogpILInRightStem[t][g-1][-1][j][0]
                                                if psplit > maxlogpILInd:
                                                    maxlogpILInd = psplit
                                            for x in range(0, d+1):
                                                for y in range(1, d-x+1):
                                                    psplit = maxlogpILInLeftStem[t][-g][-1][i+x-1][x] + maxlogpILInmed[t][g-1][j-y][d-x-y] + maxlogpILInRightStem[t][g-1][-1][j][y]

                                                    if psplit > maxlogpILInd:
                                                        maxlogpILInd = psplit

                                    maxlogpILInmedj[t][g].append(maxlogpILInd)     
                        #IL
                        if num_ILMotifs > 0:
                            tset = []
                            if num_ILMotifs > 0:
                                for t in range(0, num_ILMotifs):
                                    tset.append(maxlogpILInmedj[t][-1][d] + pIL[t])
                            maxlogpILj.append(max(tset))                        
                        #P
                        for s in range(0, d-1):
                            if s+1 < 31:
                                if searchcov(i, i+s, covlist, covpairs) == 0:
                                    psplit = maxlogpLj[d-s-1] + pP[0] + Pbulgedist[s+1-1]
                                    for nlt in RNAsequence[i:i+s+1]:
                                        psplit += bulgesing_emissionsdict[nlt]
                                    if psplit > maxlogpPd:
                                        maxlogpPd = psplit
                        for s in range(0, d-1):
                            if d-s-1 < 31:
                                if searchcov(i+s+1, j, covlist, covpairs) == 0:
                                    psplit = maxlogpL[i+s][s+1] + pP[1] + Pbulgedist[d-s-1-1]
                                    for nlt in RNAsequence[i+s+1:j+1]:
                                        psplit += bulgesing_emissionsdict[nlt]
                                    if psplit > maxlogpPd:
                                        maxlogpPd = psplit
                        if d == 2:
                            pass
                        else:
                            for x in range(i+1, j):
                                for y in range(x+1, j):
                                    if x-i < 31 and j-y < 31:
                                        if searchcov(i, x-1, covlist, covpairs) == 0 and searchcov(y+1, j, covlist, covpairs) == 0:
                                            psplit = maxlogpL[y][y-x+1] + pP[2] + Pintloopdist[x-i-1][j-y-1]
                                            for nlt in RNAsequence[i:x]:
                                                psplit += intloopsing_emissionsdict[nlt]
                                            for nlt in RNAsequence[y+1:j+1]:
                                                psplit += intloopsing_emissionsdict[nlt]
                                            if psplit > maxlogpPd:
                                                maxlogpPd = psplit
                        psplit = maxlogpHLj[d] + pP[3]
                        if psplit > maxlogpPd:
                            maxlogpPd = psplit
                        for s in range(0, d-1):
                            psplit = maxlogpM1[i+s][s+1] + maxlogpMj[d-s-1] + pP[4]
                            if psplit > maxlogpPd:
                                maxlogpPd = psplit
                        if num_ILMotifs > 0:
                            psplit = maxlogpILj[d] + pP[5]
                            if psplit > maxlogpPd:
                                maxlogpPd = psplit
                        maxlogpPj.append(maxlogpPd)
            maxlogpS.append(maxlogpSj)
            maxlogpL.append(maxlogpLj)
            maxlogpF.append(maxlogpFj)
            maxlogpP.append(maxlogpPj)
            maxlogpHL.append(maxlogpHLj)
            if num_ILMotifs > 0:
                maxlogpIL.append(maxlogpILj)
            maxlogpM.append(maxlogpMj)
            maxlogpR.append(maxlogpRj)
            maxlogpM1.append(maxlogpM1j)
            if num_HLMotifs > 0:
                for t in range(0, num_HLMotifs):
                    for g in range(0, num_HLStem[t]):
                        maxlogpHLmed[t][g].append(maxlogpHLmedj[t][g])
            if num_ILMotifs > 0:
                for t in range(0, num_ILMotifs):
                    for g in range(0, num_ILOutStem[t]):
                        maxlogpILOutmed[t][g].append(maxlogpILOutmedj[t][g])
            if num_ILMotifs > 0:
                for t in range(0, num_ILMotifs):
                    for g in range(0, num_ILInStem[t]):
                        maxlogpILInmed[t][g].append(maxlogpILInmedj[t][g])
        RNAseqlist = []
        for char in RNAsequence:
            RNAseqlist.append(char)
        parse = ["S"]
        subseq = [[n-1, n]]
        rebuild = []
        reduc = []
        for q in range(0, n):
            rebuild.append("")
            reduc.append("")
        while rebuild != RNAseqlist:
            p = len(parse)
            parsenew = []
            subseqnew = []            
            for t in range(0, p):
                j = subseq[t][0]
                d = subseq[t][1]
                i = j-d+1
                #S
                if parse[t] == "S":
                    if d == 0:
                        pass
                    else:
                        if abs(maxlogpS[j][d] - (n_emissionsdict[RNAsequence[i]] + maxlogpS[j][d-1] + pS[0])) < 10e-13:
                            parsenew.append(["n", "S"])
                            subseqnew.append([[i, 1], [j, d-1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = "."
                        for s in range(1, d+1):
                            if abs(maxlogpS[j][d] - (maxlogpL[i+s-1][s] + maxlogpS[j][d-s] + pS[1])) < 10e-13:
                                parsenew.append(["L", "S"])
                                subseqnew.append([[i+s-1, s], [j, d-s]])
                #L
                if parse[t] == "L":
                    if d < 2:
                        pass
                    else:
                        if abs(maxlogpL[j][d] - (pbpL1[bptonum(RNAsequence[i] + RNAsequence[j])] + maxlogpF[j-1][d-2] + pL[0])) < 10e-13:
                            parsenew.append(["bp(", "F", ")bp"])
                            subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = "<"
                            rebuild[j] = RNAsequence[j]
                            reduc[j] = ">"
                        elif abs(maxlogpL[j][d] - (pbpL2[bptonum(RNAsequence[i] + RNAsequence[j])] + maxlogpP[j-1][d-2] + pL[1])) < 10e-13:
                            parsenew.append(["bp(", "P", ")bp"])
                            subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = "<"
                            rebuild[j] = RNAsequence[j]
                            reduc[j] = ">"
                #F
                if parse[t] == "F":
                    if d < 2:
                        pass
                    else:
                        key2 = bptonum(RNAsequence[i-1] + RNAsequence[j+1])
                        key3 = bptonum(RNAsequence[i] + RNAsequence[j])
                        if abs(maxlogpF[j][d] - (maxlogpF[j-1][d-2] + pF[0] + stackedbpF1[key2][key3])) < 10e-13:
                            parsenew.append(["bp(", "F", ")bp"])
                            subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = "<"
                            rebuild[j] = RNAsequence[j]
                            reduc[j] = ">"
                        elif abs(maxlogpF[j][d] - (maxlogpP[j-1][d-2] + pF[1] + stackedbpF2[key2][key3])) < 10e-13:
                            parsenew.append(["bp(", "P", ")bp"])
                            subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = "<"
                            rebuild[j] = RNAsequence[j]
                            reduc[j] = ">"
                #P
                if parse[t] == "P":
                    if d == 2:
                        if abs(maxlogpP[j][d] - (maxlogpM1[i][1] + maxlogpM[j][1] + pP[4])) < 10e-13:
                            parsenew.append(["M1", "M"])
                            subseqnew.append([[i, 1], [j, 1]])
                    else:
                        for x in range(i+1, j):
                            for y in range(x+1, j):
                                if x-i < 31 and j-y < 31:
                                    psplit = maxlogpL[y][y-x+1] + pP[2] + Pintloopdist[x-i-1][j-y-1]
                                    for nlt in RNAsequence[i:x]:
                                        psplit += intloopsing_emissionsdict[nlt]
                                    for nlt in RNAsequence[y+1:j+1]:
                                        psplit += intloopsing_emissionsdict[nlt]
                                    if abs(maxlogpP[j][d] - psplit) < 10e-13:
                                        prlt = ["L"]
                                        subseqlt = []
                                        for v in range(i, x):
                                            prlt.insert(0, "n")
                                            subseqlt.append([v, 1])
                                            rebuild[v] = RNAsequence[v]
                                            reduc[v] = "."
                                        subseqlt.append([y, y-x+1])
                                        for w in range(y+1, j+1):
                                            prlt.append("n")
                                            subseqlt.append([w, 1])
                                            rebuild[w] = RNAsequence[w]
                                            reduc[w] = "."
                                        parsenew.append(prlt)
                                        subseqnew.append(subseqlt)
                    for s in range(0, d-1):
                        if s+1 < 31:
                            psplit = maxlogpL[j][d-s-1] + pP[0] + Pbulgedist[s+1-1]
                            for nlt in RNAsequence[i:i+s+1]:
                                psplit += bulgesing_emissionsdict[nlt]
                            if abs(maxlogpP[j][d] - psplit) < 10e-13:
                                prlt = ["L"]
                                subseqlt = []
                                for u in range(0, s+1):
                                    prlt.insert(0, "n")
                                    subseqlt.append([i+u, 1])
                                    rebuild[i+u] = RNAsequence[i+u]
                                    reduc[i+u] = "."
                                subseqlt.append([j, d-s-1])
                                parsenew.append(prlt)
                                subseqnew.append(subseqlt)
                    for s in range(1, d):
                        if d-s < 31:
                            psplit = maxlogpL[i+s-1][s] + pP[1] + Pbulgedist[d-s-1]
                            for nlt in RNAsequence[i+s:j+1]:
                                psplit += bulgesing_emissionsdict[nlt]
                            if abs(maxlogpP[j][d] - psplit) < 10e-13:
                                prlt = ["L"]
                                subseqlt = [[i+s-1, s]]
                                for u in range(0, d-s):
                                    prlt.append("n")
                                    subseqlt.append([i+s+u, 1])
                                    rebuild[i+s+u] = RNAsequence[i+s+u]
                                    reduc[i+s+u] = "."
                                parsenew.append(prlt)
                                subseqnew.append(subseqlt)
                    if abs(maxlogpP[j][d] - (maxlogpHL[j][d] + pP[3])) < 10e-13:
                        parsenew.append(["HL"])
                        subseqnew.append([[j, d]])
                    for s in range(1, d):
                        if abs(maxlogpP[j][d] - (maxlogpM1[i+s-1][s] + maxlogpM[j][d-s] + pP[4])) < 10e-13:
                            parsenew.append(["M1", "M"])
                            subseqnew.append([[i+s-1, s], [j, d-s]])
                    if abs(maxlogpP[j][d] - (maxlogpIL[j][d] + pP[5])) < 10e-13:
                        parsenew.append(["IL"])
                        subseqnew.append([[j, d]])
                #M
                if parse[t] == "M":
                    if d == 0:
                        pass
                    else:
                        if abs(maxlogpM[j][d] - (pM[1] + maxlogpR[j][d])) < 10e-13:
                            parsenew.append(["R"])
                            subseqnew.append([[j, d]])
                        for s in range(0, d-1):
                            if abs(maxlogpM[j][d] - (pM[0] + maxlogpM1[i+s][s+1] + maxlogpM[j][d-s-1])) < 10e-13:
                                parsenew.append(["M1", "M"])
                                subseqnew.append([[i+s, s+1], [j, d-s-1]])
                #R
                if parse[t] == "R":
                    if d == 0:
                        pass
                    else:
                        if abs(maxlogpR[j][d] - (maxlogpR[j-1][d-1] + n_emissionsdict[RNAsequence[j]] + pR[0])) < 10e-13:
                            parsenew.append(["R", "n"])
                            subseqnew.append([[j-1, d-1], [j, 1]])
                            rebuild[j] = RNAsequence[j]
                            reduc[j] = "."
                        elif abs(maxlogpR[j][d] - (maxlogpM1[j][d] + pR[1])) < 10e-13:
                            parsenew.append(["M1"])
                            subseqnew.append([[j, d]])
                #M1
                if parse[t] == "M1":
                    if d == 0:
                        pass
                    else:
                        if abs(maxlogpM1[j][d] - (n_emissionsdict[RNAsequence[i]] + maxlogpM1[j][d-1] + pM1[0])) < 10e-13:
                            parsenew.append(["n", "M1"])
                            subseqnew.append([[i, 1], [j, d-1]])
                            rebuild[i] = RNAsequence[i]
                            reduc[i] = "."
                        elif abs(maxlogpM1[j][d] - (maxlogpL[j][d] + pM1[1])) < 10e-13:
                            parsenew.append(["L"])
                            subseqnew.append([[j, d]])
                #HL
                if parse[t] == "HL":
                    if 0 < d < 33:
                        psplit = pHL[0] + HLlendist[d-1]
                        for nlt in RNAsequence[i:j+1]:
                            psplit += hlsing_emissionsdict[nlt]
                        if abs(maxlogpHL[j][d] - psplit) < 10e-13:
                            prlt = ["n"]
                            subseqlt = [[i, 1]]
                            for s in range(t+1, t+d):
                                prlt.append("n")
                                subseqlt.append([s-t+i, 1])                   
                            for s in range(i, j+1):
                                rebuild[s] = RNAsequence[s]
                                reduc[s] = "."
                            parsenew.append(prlt)
                            subseqnew.append(subseqlt)
                    if num_HLMotifs > 0:
                        for a in range(0, num_HLMotifs):
                            if abs(maxlogpHL[j][d] - (maxlogpHLmed[a][-1][j][d] + pHL[a+1])) < 10e-13:
                                parsenew.append(["HL." + str(a) + "." + str(num_HLStem[a]-1)])
                                subseqnew.append([[j, d]])
                #HLExt
                if parse[t] == "HLExt":
                    if d == 0:
                        pass
                    else:
                        parsenew.append(["n", "HLExt"])
                        subseqnew.append([[i, 1], [j, d-1]])
                        rebuild[i] = RNAsequence[i]
                        reduc[i] = "."
                
                #HLmed
                if num_HLMotifs > 0:
                    for a in range(0, num_HLMotifs):
                        for g in range(0, num_HLStem[a]):
                            if parse[t] == "HL."+str(a)+"."+str(g):
                                if g == 0:
                                    for s in range(0, d+1):
                                        psplit = maxlogpHLExt[j][d-s] + maxlogpHLloopBlocks[a][-1][i+s-1][s]
                                        if abs(maxlogpHLmed[a][g][j][d] - psplit) < 10e-13:
                                            if HLPacket[a][0] == "Bulge":
                                                parsenew.append(["HLBulge", "HLExt"])
                                            else:
                                                parsenew.append(["HLBlock " + str(a) + "." + str(len(HLPacket[a][0])), "HLExt"])
                                            subseqnew.append([[i+s-1, s], [j, d-s]])

                                else:
                                    if HLPacket[a][1][1][g-1][1] == 1:
                                            psplit = pDeleteBp(HLPacket[a][1][0][-g][0], HLPacket[a][1][1][g-1][0]) + maxlogpHLmed[a][g-1][j][d]
                                            if abs(maxlogpHLmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["HL." + str(a) + "." + str(g-1)])
                                                subseqnew.append([[j, d]])
                                            #base pair
                                            pThrough = math.log(1 - math.exp(pDeleteBp(HLPacket[a][1][0][-g][0], HLPacket[a][1][1][g-1][0])))
                                            psplit = maxlogpHLmed[a][g-1][j-1][d-2] + pBpEmit([RNAsequence[i], RNAsequence[j]], [HLPacket[a][1][0][-g][0], HLPacket[a][1][1][g-1][0]]) + pThrough    
                                            if abs(maxlogpHLmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["bp(", "HL." + str(a) + "." + str(g-1), ")bp"])
                                                subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                                                rebuild[i] = RNAsequence[i]
                                                reduc[i] = "{"
                                                rebuild[j] = RNAsequence[j]
                                                reduc[j] = "}"
                                    else:
                                        for s in range(0, d+1):
                                            psplit = maxlogpHLleftStem[a][-g][-1][i+s-1][s] + maxlogpHLmed[a][g-1][j][d-s] + maxlogpHLrightStem[a][g-1][-1][j][0]
                                            if abs(maxlogpHLmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["HLLS " + str(a) + "." + str(num_HLStem[a]-g) + "." + str(len(HLPacket[a][1][0][-g][0])), "HL." + str(a) + "." + str(g-1), "HLRS " + str(a) + "." + str(g-1) + "." + str(len(HLPacket[a][1][1][g-1][0]))])
                                                subseqnew.append([[i+s-1, s], [j, d-s], [j, 0]])                                                
                                        for x in range(0, d+1):
                                            for y in range(1, d-x+1):
                                                psplit = maxlogpHLleftStem[a][-g][-1][i+x-1][x] + maxlogpHLmed[a][g-1][j-y][d-x-y] + maxlogpHLrightStem[a][g-1][-1][j][y]
                                                if abs(maxlogpHLmed[a][g][j][d] - psplit) < 10e-13:
                                                    parsenew.append(["HLLS " + str(a) + "." + str(num_HLStem[a]-g) + "." + str(len(HLPacket[a][1][0][-g][0])), "HL." + str(a) + "." + str(g-1), "HLRS " + str(a) + "." + str(g-1) + "." + str(len(HLPacket[a][1][1][g-1][0]))])
                                                    subseqnew.append([[i+x-1, x], [j-y, d-x-y], [j, y]])  
                #HLLoop
                if num_HLMotifs > 0:
                    for a in range(0, num_HLMotifs):
                        parseHMM(RNAsequence, maxlogpIn, maxlogpHLloopBlocks[a], parsenew, subseqnew, HLPacket[a][0], j, d, t, "HLBlock " + str(a), parse, rebuild, reduc)
                        
                        
                #HLBulge
                if parse[t] == "HLBulge":
                    prlt = ["n"]
                    subseqlt = [[i, 1]]
                    for s in range(t+1, t+d):
                        prlt.append("n")
                        subseqlt.append([s-t+i, 1])                   
                    for s in range(i, j+1):
                        rebuild[s] = RNAsequence[s]
                        reduc[s] = "."
                    parsenew.append(prlt)
                    subseqnew.append(subseqlt)
                
                #HLStem
                if num_HLMotifs > 0:
                    for a in range(0, num_HLMotifs):
                        for h in range(0, num_HLStem[a]):
                            parseHMM(RNAsequence, maxlogpIn, maxlogpHLleftStem[a][h], parsenew, subseqnew, HLPacket[a][1][0][h][0], j, d, t, "HLLS " + str(a) + "." + str(h), parse, rebuild, reduc)
                            parseHMM(RNAsequence, maxlogpIn, maxlogpHLrightStem[a][h], parsenew, subseqnew, HLPacket[a][1][1][h][0], j, d, t, "HLRS " + str(a) + "." + str(h), parse, rebuild, reduc)
                #IL
                if num_ILMotifs > 0:
                    if parse[t] == "IL":
                        for a in range(0, num_ILMotifs):
                            if abs(maxlogpIL[j][d] - (pIL[a] + maxlogpILInmed[a][-1][j][d])) < 10e-13:
                                parsenew.append(["IL.In."+str(a)+"."+str(num_ILInStem[a]-1)])
                                subseqnew.append([[j, d]])
                        
                #ILInmed
                if num_ILMotifs > 0:
                    for a in range(0, num_ILMotifs):
                        for g in range(0, num_ILInStem[a]):
                            if parse[t] == "IL.In."+str(a)+"."+str(g):
                                
                                if g == 0:
                                    if num_ILOutStem[a] == 0:
                                        for s in range(0, d+1):
                                            psplit = maxlogpILLeft[a][-1][i+s-1][s] + maxlogpL[j][d-s] + maxlogpILRight[a][-1][j][0]
                                            if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["ILLBulge " + str(a) + "." + str(len(ILPacket[a][1][0])), "L", "ILRBulge " + str(a) + "." + str(len(ILPacket[a][1][1]))])
                                                subseqnew.append([[i+s-1, s], [j, d-s], [j, 0]])      
                                                
                                                
                                        for x in range(0, d+1):
                                            for y in range(1, d-x+1):
                                                psplit = maxlogpILLeft[a][-1][i+x-1][x] + maxlogpL[j-y][d-x-y] + maxlogpILRight[a][-1][j][y]
                                                if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                                    parsenew.append(["ILLBulge " + str(a) + "." + str(len(ILPacket[a][1][0])), "L", "ILRBulge " + str(a) + "." + str(len(ILPacket[a][1][1]))])
                                                    subseqnew.append([[i+x-1, x], [j-y, d-x-y], [j, y]]) 
                                                          
                                    else:
                                        for s in range(0, d+1):
                                            psplit = maxlogpILLeft[a][-1][i+s-1][s] + maxlogpILOutmed[a][-1][j][d-s] + maxlogpILRight[a][-1][j][0]
                                            if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["ILLBulge " + str(a) + "." + str(len(ILPacket[a][1][0])), "IL.Out."+str(a)+"."+str(num_ILOutStem[a]-1), "ILRBulge " + str(a) + "." + str(len(ILPacket[a][1][1]))])
                                                subseqnew.append([[i+s-1, s], [j, d-s], [j, 0]]) 
                                                
                                                
                                        for x in range(0, d+1):
                                            for y in range(1, d-x+1):
                                                psplit = maxlogpILLeft[a][-1][i+x-1][x] + maxlogpILOutmed[a][-1][j-y][d-x-y] + maxlogpILRight[a][-1][j][y]
                                                if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                                    parsenew.append(["ILLBulge " + str(a) + "." + str(len(ILPacket[a][1][0])), "IL.Out."+str(a)+"."+str(num_ILOutStem[a]-1), "ILRBulge " + str(a) + "." + str(len(ILPacket[a][1][1]))])
                                                    subseqnew.append([[i+x-1, x], [j-y, d-x-y], [j, y]]) 
                                                    
                                else:
                                    
                                    if ILPacket[a][0][1][g-1][1] == 1:
                                        
                                        psplit = pDeleteBp(ILPacket[a][0][0][-g][0], ILPacket[a][0][1][g-1][0]) + maxlogpILInmed[a][g-1][j][d]
                                        
                                        if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                            parsenew.append(["IL.In."+str(a)+"."+str(g-1)])
                                            subseqnew.append([[j, d]])
                                        pThrough = math.log(1 - math.exp(pDeleteBp(ILPacket[a][0][0][-g][0], ILPacket[a][0][1][g-1][0])))
                                        psplit = maxlogpILInmed[a][g-1][j-1][d-2] + pThrough + pBpEmit([RNAsequence[i], RNAsequence[j]], [ILPacket[a][0][0][-g][0], ILPacket[a][0][1][g-1][0]])
                                        key1 = bptonum(RNAsequence[i] + RNAsequence[j])
                                        if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                            parsenew.append(["bp(", "IL.In." + str(a) + "." + str(g-1), ")bp"])
                                            subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                                            rebuild[i] = RNAsequence[i]
                                            reduc[i] = "{"
                                            rebuild[j] = RNAsequence[j]
                                            reduc[j] = "}"   
                                    else:
                                        for s in range(0, d+1):
                                            psplit = maxlogpILInLeftStem[a][-g][-1][i+s-1][s] + maxlogpILInmed[a][g-1][j][d-s] + maxlogpILInRightStem[a][g-1][-1][j][0]
                                            
                                            if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["IL_In_LS " + str(a) + "." + str(num_ILInStem[a] - g) + "." + str(len(ILPacket[a][0][0][-g][0])), "IL.In." + str(a) + "." + str(g-1), "IL_In_RS " + str(a) + "." + str(g-1) + "." + str(len(ILPacket[a][0][1][g-1][0]))])
                                                subseqnew.append([[i+s-1, s], [j, d-s], [j, 0]])                                                
                                                
                                        for x in range(0, d+1):
                                            for y in range(1, d-x+1):
                                                psplit = maxlogpILInLeftStem[a][-g][-1][i+x-1][x] + maxlogpILInmed[a][g-1][j-y][d-x-y] + maxlogpILInRightStem[a][g-1][-1][j][y]
                                                
                                                if abs(maxlogpILInmed[a][g][j][d] - psplit) < 10e-13:
                                                    parsenew.append(["IL_In_LS " + str(a) + "." + str(num_ILInStem[a] - g) + "." + str(len(ILPacket[a][0][0][-g][0])), "IL.In." + str(a) + "." + str(g-1), "IL_In_RS " + str(a) + "." + str(g-1) + "." + str(len(ILPacket[a][0][1][g-1][0]))])
                                                    subseqnew.append([[i+x-1, x], [j-y, d-x-y], [j, y]])                                
                
                #IlOutmed
                if num_ILMotifs > 0:
                    for a in range(0, num_ILMotifs):
                        for g in range(0, num_ILOutStem[a]):
                            if parse[t] == "IL.Out."+str(a)+"."+str(g):
                                if g == 0:
                                    #base pair when returning to L
                                    if ILPacket[a][2][1][g][1] == 1:
                                        psplit = pDeleteBp(ILPacket[a][2][0][-1-g][0], ILPacket[a][2][1][g][0]) + maxlogpL[j][d]
                                        if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                            parsenew.append(["L"])
                                            subseqnew.append([[j, d]])                                       
                                        else:    
                                            pThrough = math.log((1 - math.exp(pDeleteBp(ILPacket[a][2][0][-1-g][0], ILPacket[a][2][1][g][0]))))
                                            psplit = maxlogpL[j-1][d-2] + pBpEmit([RNAsequence[i], RNAsequence[j]], [ILPacket[a][2][0][-1-g][0], ILPacket[a][2][1][g][0]]) + pThrough
                                            if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["bp(", "L", ")bp"])
                                                subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                                                rebuild[i] = RNAsequence[i]
                                                reduc[i] = "{"
                                                rebuild[j] = RNAsequence[j]
                                                reduc[j] = "}"                                                
                                    else:
                                        for s in range(0, d+1):
                                            psplit = maxlogpILOutLeftStem[a][-g-1][-1][i+s-1][s] + maxlogpL[j][d-s] + maxlogpILOutRightStem[a][g][-1][j][0]
                                            if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["IL_Out_LS "+str(a)+"."+str(num_ILOutStem[a]-g-1) + "." +str(len(ILPacket[a][2][0][-1-g][0])), "L", "IL_Out_RS "+str(a)+"."+str(g) + "." +str(len(ILPacket[a][2][1][g][0]))])
                                                subseqnew.append([[i+s-1, s], [j, d-s], [j, 0]])
                                        for x in range(0, d+1):
                                            for y in range(1, d-x+1):
                                                psplit = maxlogpILOutLeftStem[t][-g-1][-1][i+x-1][x] + maxlogpL[j-y][d-x-y] + maxlogpILOutRightStem[t][g][-1][j][y]
                                                if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                                    parsenew.append(["IL_Out_LS "+str(a)+"."+str(num_ILOutStem[a]-g-1) + "." +str(len(ILPacket[a][2][0][-1-g][0])), "L", "IL_Out_RS "+str(a)+"."+str(g) + "." +str(len(ILPacket[a][2][1][g][0]))])
                                                    subseqnew.append([[i+x-1, x], [j-y, d-x-y], [j, y]])
                                else:
                                    if ILPacket[a][2][1][g][1] == 1:
                                        psplit = pDeleteBp(ILPacket[a][2][0][-1-g][0], ILPacket[a][2][1][g][0]) + maxlogpILOutmed[a][g-1][j][d]
                                        if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                            parsenew.append(["IL.Out."+str(a)+"."+str(g-1)])
                                            subseqnew.append([[j, d]])
                                        pThrough = math.log(1 - math.exp(pDeleteBp(ILPacket[a][2][0][-1-g][0], ILPacket[a][2][1][g][0])))
                                        psplit = maxlogpILOutmed[a][g-1][j-1][d-2] + pBpEmit([RNAsequence[i], RNAsequence[j]], [ILPacket[a][2][0][-1-g][0], ILPacket[a][2][1][g][0]]) + pThrough
                                        if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                            parsenew.append(["bp(", "IL.Out."+str(a)+"."+str(g-1), ")bp"])
                                            subseqnew.append([[i, 1], [j-1, d-2], [j, 1]])
                                            rebuild[i] = RNAsequence[i]
                                            reduc[i] = "{"
                                            rebuild[j] = RNAsequence[j]
                                            reduc[j] = "}" 
                                            
                                    else:
                                        for s in range(0, d+1):
                                            psplit = maxlogpILOutLeftStem[a][-g-1][-1][i+s-1][s] + maxlogpILOutmed[a][g-1][j][d-s] + maxlogpILOutRightStem[a][g][-1][j][0]
                                            if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                                parsenew.append(["IL_Out_LS "+str(a)+"."+str(num_ILOutStem[a]-g-1) + "." +str(len(ILPacket[a][2][0][-1-g][0])), "IL.Out."+str(a)+"."+str(g-1), "IL_Out_RS "+str(a)+"."+str(g) + "." +str(len(ILPacket[a][2][1][g][0]))])
                                                subseqnew.append([[i+s-1, s], [j, d-s], [j, 0]])
                                        for x in range(0, d+1):
                                            for y in range(1, d-x+1):
                                                psplit = maxlogpILOutLeftStem[a][-g-1][-1][i+x-1][x] + maxlogpILOutmed[a][g-1][j-y][d-x-y] + maxlogpILOutRightStem[a][g][-1][j][y]
                                                if abs(maxlogpILOutmed[a][g][j][d] - psplit) < 10e-13:
                                                    parsenew.append(["IL_Out_LS "+str(a)+"."+str(num_ILOutStem[a]-g-1) + "." +str(len(ILPacket[a][2][0][-1-g][0])), "IL.Out."+str(a)+"."+str(g-1), "IL_Out_RS "+str(a)+"."+str(g) + "." +str(len(ILPacket[a][2][1][g][0]))])
                                                    subseqnew.append([[i+x-1, x], [j-y, d-x-y], [j, y]])
                
                
                #ILStem (In/Out)
                #In
                if num_ILMotifs > 0:
                    for a in range(0, (num_ILMotifs)):
                        for h in range(0, num_ILInStem[a]-1):
                            
                            parseHMM(RNAsequence, maxlogpIn, maxlogpILInLeftStem[a][h], parsenew, subseqnew, ILPacket[a][0][0][h][0], j, d, t, "IL_In_LS " + str(a) + "." + str(h+1), parse, rebuild, reduc)
                            parseHMM(RNAsequence, maxlogpIn, maxlogpILInRightStem[a][h], parsenew, subseqnew, ILPacket[a][0][1][h][0], j, d, t, "IL_In_RS " + str(a) + "." + str(h), parse, rebuild, reduc)
                            
                #Out
                if num_ILMotifs > 0:
                    for a in range(0, (num_ILMotifs)):
                        for h in range(0, num_ILOutStem[a]):
                            
                            parseHMM(RNAsequence, maxlogpIn, maxlogpILOutLeftStem[a][h], parsenew, subseqnew, ILPacket[a][2][0][h][0], j, d, t, "IL_Out_LS " + str(a) + "." + str(h), parse, rebuild, reduc)
                            parseHMM(RNAsequence, maxlogpIn, maxlogpILOutRightStem[a][h], parsenew, subseqnew, ILPacket[a][2][1][h][0], j, d, t, "IL_Out_RS " + str(a) + "." + str(h), parse, rebuild, reduc)

                
                #Loop HMMS (Bulges too)
                if num_ILMotifs > 0:
                    for a in range(0, (num_ILMotifs)):
                        
                        parseHMM(RNAsequence, maxlogpIn, maxlogpILLeft[a], parsenew, subseqnew, ILPacket[a][1][0], j, d, t, "ILLBulge " + str(a), parse, rebuild, reduc)
                        parseHMM(RNAsequence, maxlogpIn, maxlogpILRight[a], parsenew, subseqnew, ILPacket[a][1][1], j, d, t, "ILRBulge " + str(a), parse, rebuild, reduc)
                        
                #ILBulge
                if parse[t] == "ILBulge":
                    prlt = ["n"]
                    subseqlt = [[i, 1]]
                    for s in range(t+1, t+d):
                        prlt.append("n")
                        subseqlt.append([s-t+i, 1])                   
                    for s in range(i, j+1):
                        rebuild[s] = RNAsequence[s]
                        reduc[s] = "."
                    parsenew.append(prlt)
                    subseqnew.append(subseqlt)
                
                #Insert
                if parse[t] == "In":
                    if d == 0:
                        pass
                    elif d == 1:
                        parsenew.append(["n"])
                        subseqnew.append([[j, d]])
                        rebuild[j] = RNAsequence[j]
                        reduc[j] = "x"
                    else:
                        parsenew.append(["n", "In"])
                        subseqnew.append([[i, 1], [j, d-1]])
                        rebuild[i] = RNAsequence[i]
                        reduc[i] = "x" 
            b = []
            for el in parsenew:
                for el1 in el:
                    b.append(el1)
            parse = b
            c = []
            for el in subseqnew:
                for el1 in el:
                    c.append(el1)
            subseq = c
    return listtostr(reduc)

In [73]:
seq = "AUGACCAGGCUUCAAAGGAUCUUGUCCGUUUAGUUUGUGUCGCCAGAAUGUUAUAAAAUUAUGACCCUGGGCAAAAGUUCGAUGACGUAUUGAUCAUAUUACCAUUUCGAGCGUGAACAAGGCAAAAGCUGUUGAGUCUUACUCAACUUGGCAUUUGUUUAUAAUGAUCGCUUAUUGAUUCUCAUUUGGUCAAUGAAGGAAUGGUAAUAUUUAUCAGAGGACU"

pairs = [[95, 209], [96, 208], [97, 207], [98, 206], [99, 205], [100, 204], [102, 202], [103, 201], [104, 200], [131, 145], [132, 144], [133, 143], [134, 142], [173, 193], [174, 192], [177, 189]]


In [74]:
parsesequence(seq, pairs, [])

'....<<<<<..........................<<......>>....................>>>>>.....<<<<<rnnga{<........<<<<<<<<<<<<<<<<<<{ga<...<<<<<<.<<<<<<<<<.....>>>>>>..>>>.>>>>>>..>rnnga}>>>>><<<<<<.........>>>>>>...>>>>>>>>>>>>>....>}ga>>>>>'

In [100]:
seq = "GGGUGCGAUCAUACCAGCGUUAAUGCACCGGAUCCCAUCAGAACUCCGCAGUUAAGCGCGCUUGGGCUAGAGUAGUACUGGGAUGGGUGACCUCCCGGGAAGUCCUAGUGUUGCACCC"
cpairs = [[0, 117], [1, 116], [2, 115], [3, 114], [4, 113], [7, 110], [13, 64], [14, 63], [15, 61], [16, 60], [17, 59], [18, 58], [19, 57], [26, 51], [28, 47], [29, 46], [30, 45], [31, 44], [36, 42], [52, 55], [66, 107], [67, 106], [68, 105], [69, 104], [70, 103], [77, 97], [78, 96], [80, 94], [81, 93], [82, 92], [84, 91], [85, 90]]


In [101]:
parsesequence(seq, cpairs, [])

'<<<<<<<<<....<<<<<<<......<<<<<<...<<.....>>>>>>..>><..>.>>>>>.>><<<<<<<.....<<<<<<.<<gnra>>>>>>>>....>>>>>>>>>>>>>>>>'

In [176]:
seq = "UACCUCGCGACAGGGGCAAUAUAGCAGCAAGUGACGGUUAACUGAUGCGCUAUUAUUGCUAGUUGAAAACUACUUCAAUAAGUGGAAACGACGCUUGCGUCGGGUCCCAAUUUCUGGAAGGUCGUAUGAC"

pairs = [[20, 52], [21, 51], [22, 50], [23, 49], [24, 48], [26, 47], [27, 46], [28, 45], [34, 42], [36, 40], [82, 107], [83, 106], [84, 105], [85, 104], [86, 103], [87, 102], [89, 100], [90, 99], [91, 98], [92, 97], [120, 129], [121, 128], [122, 127], [123, 126]]
